# BitNet latent reasoning: compressed feedback vs pause tokens vs CoT

**Question.** Coconut-style latent reasoning with a *compressor*: run the BitNet b1.58 LLM over a prompt, read the
hidden states at layer *i* (default ≈ 2/3 depth), compress them with K learned queries (Perceiver-style cross-attention)
into **K soft tokens**, append those as input embeddings (re-using the KV cache, so only the K new positions are
computed), repeat **R** times, then decode the answer. Does feeding back compressed layer-*i* content beat the *same
extra compute without feedback* (learned **pause** tokens)?

**Task.** Synthetic variable tracking (`a = 3`, `b = a + 2`, ..., `print(d)`) with `hops` = chain length to the queried
variable, interleaved distractor assignments, values in [0, 99]. Trained on hops 2–4 (QUICK) / 2–6 (FULL), tested on
longer chains too (hops beyond the training range) and with an **eval-R sweep** (test-time scaling: more thought steps
at test time than in training).

**Modes** (matched settings: same base model, LoRA rank, steps, batch, data):
| mode | what is between the prompt and `Answer:` | loss |
|---|---|---|
| `none` | nothing | answer |
| `cot` | the explicit trace (`b = 5`, `c = 20`, ...) as text | trace + answer |
| `pause` | R×K learned constant pause embeddings (no feedback) | answer (+ remaining trace during curriculum) |
| `latent` | R steps × K compressor soft tokens (feedback of layer-*i* states) | answer (+ remaining trace during curriculum) |

`pause` and `latent` use the Coconut curriculum: stage *s* replaces the first *s* trace lines by *s* thought steps;
the last stage is all-latent with R steps.

The base model (`microsoft/bitnet-b1.58-2B-4T-bf16`, public, MIT, no token needed) is frozen and runs the BitNet
W1.58A8 forward (ternary absmean weights, per-token int8 activations, as in bitnet.cpp); LoRA adapters (full
precision) are added on q/k/v/o/gate/up/down, and the compressor is full precision.

**Runtime.** L4 or A100 with `bf16` is recommended. A T4 (16 GB) works with emulated `bf16` (measured: 6.1 GB peak, ~1.2–1.5 examples/s;
`fp16` overflows and gives NaN loss with 2B-4T) but is several times slower.
Runtimes below are *estimates* (not measured on Colab): QUICK ≈ 1 h on L4 for the four modes, FULL ≈ 5–7 h on L4
(≈ 3 h on A100).

In [ ]:
!nvidia-smi || echo "no GPU (the tiny sanity run still works on CPU)"
# torch comes with Colab; the code was verified with transformers==5.0.0 (+ torch 2.9).
!pip install -q "transformers==5.0.0" safetensors pytest matplotlib

## Sources
Default: clone the public fork `eduardonery1/BitNet` (branch `feature/bitnet-vl`) and work in `BitNet/multimodal`.

In [ ]:
# Default: get the sources from the public fork (no --recursive: the latent experiment does not need the submodule).
!test -d BitNet || git clone -b feature/bitnet-vl --depth 1 https://github.com/eduardonery1/BitNet
%cd BitNet/multimodal

### Fallback: embedded sources (skipped by default)
Only used when `USE_EMBEDDED_SOURCES = True` or the clone is missing the latent experiment. Contains `multimodal/bitnet_vl` (the files the latent experiment imports) and `bitnet_vl/latent/`.

In [ ]:
# FALLBACK (skipped by default): embedded copies of the sources, written only if USE_EMBEDDED_SOURCES = True or the
# clone above did not provide them (e.g. the pushed branch is stale). They are the files this notebook was built from.
import os, sys
USE_EMBEDDED_SOURCES = False
_have_clone = os.path.exists("bitnet_vl/latent/reasoner.py")
EMBEDDED_SOURCES = {
    'bitnet_vl/__init__.py': '"""BitNet-VL: ternary vision-language model (PyTorch reference / training code). See docs/bitnet-vl-design.md."""\n\nfrom .bitlinear import BitLinear, replace_linears, ternary_weight\nfrom .modeling import (\n    BitNetVLConfig,\n    BitNetVLForConditionalGeneration,\n    pixel_shuffle,\n    pixel_shuffle_reference,\n)\n\n__all__ = [\n    "BitLinear",\n    "replace_linears",\n    "ternary_weight",\n    "BitNetVLConfig",\n    "BitNetVLForConditionalGeneration",\n    "pixel_shuffle",\n    "pixel_shuffle_reference",\n]\n',
    'bitnet_vl/bitlinear.py': '"""BitLinear (W1.58 A8) for quantization-aware training.\n\nForward semantics (identical to what bitnet.cpp\'s I2_S `ggml_mul_mat` computes at inference):\n\n* weights: per-tensor absmean ternary. ``s = mean(|W|).clamp(min=1e-5)``; ``W_q = round(W / s).clamp(-1, 1) * s``\n  (this is exactly ``BitnetModel.weight_quant`` in ``utils/convert-hf-to-gguf-bitnet.py``).\n* activations: per-token absmax int8. ``a = 127 / max(|x|, dim=-1).clamp(min=1e-5)``;\n  ``x_q = round(x * a).clamp(-128, 127) / a``.\n* gradients: straight-through estimator, ``x + (q(x) - x).detach()``, so the latent full-precision weights receive\n  the gradient of the quantized forward.\n\n`BitLinear` subclasses `nn.Linear`, so the parameter names (``weight``, ``bias``) and state-dict keys are unchanged\nand a model with BitLinear layers saves/loads exactly like the original HF model.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Callable\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nEPS = 1e-5\n\n\ndef weight_quant(w: torch.Tensor) -> torch.Tensor:\n    """Fake-quantize weights to {-s, 0, +s} (no STE)."""\n    dtype = w.dtype\n    w = w.float()\n    s = 1.0 / w.abs().mean().clamp(min=EPS)\n    return ((w * s).round().clamp(-1, 1) / s).to(dtype)\n\n\ndef activation_quant(x: torch.Tensor) -> torch.Tensor:\n    """Fake-quantize activations per token (last dim) to int8 levels (no STE)."""\n    dtype = x.dtype\n    x = x.float()\n    s = 127.0 / x.abs().amax(dim=-1, keepdim=True).clamp(min=EPS)\n    return ((x * s).round().clamp(-128, 127) / s).to(dtype)\n\n\ndef ternary_weight(w: torch.Tensor) -> tuple[torch.Tensor, float]:\n    """Export helper: returns (int8 tensor with values in {-1, 0, 1}, scale) with ``w ~= q * scale``.\n\n    ``scale = mean(|w|).clamp(min=1e-5)`` -- the same as the converter\'s ``weight_quant`` / ``quantize_to_i2_s``.\n    """\n    w = w.detach().float()\n    inv = 1.0 / w.abs().mean().clamp(min=EPS)  # same op order as the converter\'s weight_quant\n    q = (w * inv).round().clamp(-1, 1).to(torch.int8)\n    return q, float(1.0 / inv)\n\n\nclass BitLinear(nn.Linear):\n    """Drop-in nn.Linear with ternary-weight / int8-activation fake quantization and STE gradients."""\n\n    def __init__(self, in_features: int, out_features: int, bias: bool = True, device=None, dtype=None,\n                 quantize_activations: bool = True):\n        super().__init__(in_features, out_features, bias=bias, device=device, dtype=dtype)\n        self.quantize_activations = quantize_activations\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        w = self.weight\n        w_q = w + (weight_quant(w) - w).detach()\n        if self.quantize_activations:\n            x = x + (activation_quant(x) - x).detach()\n        return F.linear(x, w_q, self.bias)\n\n    @classmethod\n    def from_linear(cls, linear: nn.Linear) -> "BitLinear":\n        new = cls(linear.in_features, linear.out_features, bias=linear.bias is not None,\n                  device=linear.weight.device, dtype=linear.weight.dtype)\n        # share the parameters: no copy, preserves requires_grad and optimizer identity\n        new.weight = linear.weight\n        if linear.bias is not None:\n            new.bias = linear.bias\n        return new\n\n    def extra_repr(self) -> str:\n        return super().extra_repr() + ", w=ternary(absmean), a=int8(absmax/token)"\n\n\ndef replace_linears(module: nn.Module, predicate: Callable[[str, nn.Linear], bool] | None = None) -> list[str]:\n    """Swap every ``nn.Linear`` submodule (for which ``predicate(qualified_name, linear)`` is true) with a\n    `BitLinear` that shares the same weight/bias Parameters. Returns the list of replaced names. Idempotent:\n    existing BitLinear modules are left alone.\n\n    Subclasses of nn.Linear that carry their own quantization (e.g. transformers\' ``AutoBitLinear``) are also\n    replaced; ones with extra buffers such as an offline ``weight_scale`` are rejected because their weights are\n    packed, not latent.\n    """\n    replaced = []\n    for name, sub in list(module.named_modules()):\n        if not isinstance(sub, nn.Linear) or isinstance(sub, BitLinear):\n            continue\n        if predicate is not None and not predicate(name, sub):\n            continue\n        if any(n == "weight_scale" for n, _ in sub.named_buffers(recurse=False)):\n            raise ValueError(f"{name}: offline-quantized (packed) linear cannot be used for QAT; "\n                             "load the bf16 master-weight checkpoint instead")\n        if getattr(sub, "rms_norm", None) is not None:\n            raise ValueError(f"{name}: AutoBitLinear with internal rms_norm is not supported")\n        module.set_submodule(name, BitLinear.from_linear(sub))\n        replaced.append(name)\n    return replaced\n',
    'bitnet_vl/modeling.py': '"""BitNet-VL: SigLIP vision tower -> 2x2 pixel shuffle -> FP projector -> BitNet b1.58 LLM.\n\nSee docs/bitnet-vl-design.md for the binding decisions. Things the C++ (clip.cpp / mtmd) side must replicate exactly:\n\n1. Vision features = ``SiglipVisionModel(...).last_hidden_state``, i.e. the output of the LAST encoder layer followed\n   by ``post_layernorm``. The attention-pooling head is not used (built with ``vision_use_head=False``, so no\n   ``vision_model.head.*`` keys are saved).\n2. Pixel shuffle (factor s=2, SmolVLM / Idefics3 ordering), see `pixel_shuffle` / `pixel_shuffle_reference`:\n   input  x[b, h*W + w, c]          (h, w in the H x W = 24 x 24 patch grid, row-major, c < D = 768)\n   output y[b, i*(W/s) + j, (dy*s + dx)*D + c] = x[b, (s*i + dy)*W + (s*j + dx), c]\n   for i < H/s, j < W/s, dy, dx < s. So channel block k = dy*s + dx is the patch at row offset dy, column offset dx\n   (blocks ordered (0,0), (0,1), (1,0), (1,1)) and output tokens are row-major over the 12 x 12 grid.\n3. Projector: LayerNorm(s*s*D, eps=1e-5) -> Linear(s*s*D, H_llm) -> GELU (exact erf form, ``nn.GELU()``) ->\n   Linear(H_llm, H_llm). All with bias, full precision.\n4. Prompt: ``<|begin_of_text|>User: <image>\\n{question}<|eot_id|>Assistant: `` and the single ``<image>`` token is\n   replaced by the num_image_tokens projected embeddings (in order).\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport json\nimport os\nimport re\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\n\nimport torch\nimport torch.nn as nn\nfrom safetensors.torch import load_file, save_file\n\nfrom .bitlinear import BitLinear, replace_linears\nfrom .tokenizer_utils import EOT_TOKEN, IMAGE_TOKEN, add_image_token\n\nLLM_LINEAR_RE = re.compile(r"(^|\\.)layers\\.\\d+\\.(self_attn\\.(q|k|v|o)_proj|mlp\\.(gate|up|down)_proj)$")\nVISION_LINEAR_RE = re.compile(r"(^|\\.)encoder\\.layers\\.\\d+\\.(self_attn\\.(q|k|v|out)_proj|mlp\\.fc[12])$")\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# config\n# --------------------------------------------------------------------------------------------------------------------\n@dataclass\nclass BitNetVLConfig:\n    # --- contract keys (docs/bitnet-vl-design.md) ---\n    model_type: str = "bitnet_vl"\n    vision_model_name: str = "google/siglip-base-patch16-384"\n    llm_model_name: str = "microsoft/bitnet-b1.58-2B-4T-bf16"\n    pixel_shuffle: int = 2\n    vision_ternary: bool = False\n    image_size: int = 384\n    num_image_tokens: int = 144\n    llm_hidden: int = 2560\n    # --- additional keys (informative for the exporter; never required by the contract) ---\n    vision_hidden: int = 768\n    patch_size: int = 16\n    projector_input: int = 3072\n    projector_act: str = "gelu"  # exact (erf) GELU\n    projector_norm_eps: float = 1e-5\n    vision_feature: str = "last_hidden_state_post_layernorm"\n    pixel_shuffle_order: str = "idefics3"  # y[i*(W/s)+j, (dy*s+dx)*D + c] = x[(s*i+dy)*W + s*j+dx, c]\n    image_mean: list = field(default_factory=lambda: [0.5, 0.5, 0.5])\n    image_std: list = field(default_factory=lambda: [0.5, 0.5, 0.5])\n    image_token: str = IMAGE_TOKEN\n    image_token_id: int = -1\n    llm_qat: bool = True\n    vision_config: dict = field(default_factory=dict)  # SiglipVisionConfig.to_dict()\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, d: dict) -> "BitNetVLConfig":\n        known = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}\n        return cls(**known)\n\n    def save(self, path: str | os.PathLike):\n        with open(path, "w", encoding="utf-8") as f:\n            json.dump(self.to_dict(), f, indent=2)\n\n    @classmethod\n    def load(cls, path: str | os.PathLike) -> "BitNetVLConfig":\n        with open(path, encoding="utf-8") as f:\n            return cls.from_dict(json.load(f))\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# pixel shuffle\n# --------------------------------------------------------------------------------------------------------------------\ndef pixel_shuffle(x: torch.Tensor, s: int = 2) -> torch.Tensor:\n    """(B, H*W, D) -> (B, H*W/s^2, D*s^2), Idefics3/SmolVLM ordering (see module docstring)."""\n    b, n, d = x.shape\n    h = w = int(round(n ** 0.5))\n    assert h * w == n and h % s == 0, f"need a square grid divisible by {s}, got {n} tokens"\n    x = x.view(b, h, w // s, d * s)              # [b, h, j, dx*D + c]\n    x = x.permute(0, 2, 1, 3)                    # [b, j, h, dx*D + c]\n    x = x.reshape(b, w // s, h // s, d * s * s)  # [b, j, i, dy*s*D + dx*D + c]\n    x = x.permute(0, 2, 1, 3)                    # [b, i, j, ...]\n    return x.reshape(b, (h // s) * (w // s), d * s * s)\n\n\ndef pixel_shuffle_reference(x: torch.Tensor, s: int = 2) -> torch.Tensor:\n    """Loop reference of `pixel_shuffle`: the formula the C++ side must implement."""\n    b, n, d = x.shape\n    h = w = int(round(n ** 0.5))\n    y = x.new_empty(b, (h // s) * (w // s), d * s * s)\n    for i in range(h // s):\n        for j in range(w // s):\n            for dy in range(s):\n                for dx in range(s):\n                    k = dy * s + dx\n                    y[:, i * (w // s) + j, k * d:(k + 1) * d] = x[:, (s * i + dy) * w + (s * j + dx), :]\n    return y\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# projector\n# --------------------------------------------------------------------------------------------------------------------\nclass Projector(nn.Module):\n    def __init__(self, in_dim: int, out_dim: int, eps: float = 1e-5):\n        super().__init__()\n        self.norm = nn.LayerNorm(in_dim, eps=eps)\n        self.fc1 = nn.Linear(in_dim, out_dim)\n        self.act = nn.GELU()\n        self.fc2 = nn.Linear(out_dim, out_dim)\n\n    def forward(self, x):\n        return self.fc2(self.act(self.fc1(self.norm(x))))\n\n\n@dataclass\nclass BitNetVLOutput:\n    loss: torch.Tensor | None = None\n    logits: torch.Tensor | None = None\n    vision_hidden_states: tuple | None = None  # per-encoder-layer outputs (len = num_layers), if requested\n    labels: torch.Tensor | None = None  # expanded labels aligned with logits\n\n\ndef _strip_quant_config(cfg):\n    if getattr(cfg, "quantization_config", None) is not None:\n        # HF\'s bitnet quantizer would swap in AutoBitLinear (torch.compile\'d online quant or packed offline\n        # weights). We do our own QAT with BitLinear on the plain latent weights instead.\n        cfg.quantization_config = None\n        try:\n            delattr(cfg, "quantization_config")\n        except AttributeError:\n            pass\n    return cfg\n\n\ndef _ensure_rope_theta(llm_dir: Path):\n    """transformers 5 stores rope_theta inside rope_parameters; the bitnet GGUF converter reads the top-level key."""\n    p = llm_dir / "config.json"\n    with open(p, encoding="utf-8") as f:\n        cfg = json.load(f)\n    rp = cfg.get("rope_parameters") or {}\n    if "rope_theta" not in cfg and "rope_theta" in rp:\n        cfg["rope_theta"] = rp["rope_theta"]\n    cfg.pop("quantization_config", None)\n    with open(p, "w", encoding="utf-8") as f:\n        json.dump(cfg, f, indent=2)\n\n\n# --------------------------------------------------------------------------------------------------------------------\n# model\n# --------------------------------------------------------------------------------------------------------------------\nclass BitNetVLForConditionalGeneration(nn.Module):\n    def __init__(self, config: BitNetVLConfig, vision_tower: nn.Module, language_model: nn.Module, tokenizer=None):\n        super().__init__()\n        self.config = config\n        self.vision_tower = vision_tower\n        self.language_model = language_model\n        self.tokenizer = tokenizer\n        vc = vision_tower.config\n        s = config.pixel_shuffle\n        config.vision_hidden = vc.hidden_size\n        config.patch_size = vc.patch_size\n        config.image_size = vc.image_size\n        config.projector_input = vc.hidden_size * s * s\n        config.llm_hidden = language_model.config.hidden_size\n        config.num_image_tokens = (vc.image_size // vc.patch_size // s) ** 2\n        config.vision_config = vc.to_dict()\n        self.projector = Projector(config.projector_input, config.llm_hidden, eps=config.projector_norm_eps)\n        if tokenizer is not None:\n            tid = add_image_token(tokenizer)\n            config.image_token_id = tid\n            if language_model.get_input_embeddings().num_embeddings < len(tokenizer):\n                language_model.resize_token_embeddings(len(tokenizer), mean_resizing=False)\n        if config.llm_qat:\n            self.quantize_llm_()\n        if config.vision_ternary:\n            self.ternarize_vision_()\n\n    # ---------------------------------------------------------------- construction\n    @classmethod\n    def from_configs(cls, vision_config, llm_config, tokenizer=None, **cfg_kwargs):\n        """Randomly initialized model (tests / tiny checkpoints)."""\n        from transformers import BitNetForCausalLM, SiglipVisionModel\n\n        vision_config.vision_use_head = False\n        vt = SiglipVisionModel(vision_config)\n        llm = BitNetForCausalLM(_strip_quant_config(llm_config))\n        return cls(BitNetVLConfig(**cfg_kwargs), vt, llm, tokenizer)\n\n    @classmethod\n    def from_components(cls, vision_model_name: str, llm_model_name: str, dtype=torch.float32, **cfg_kwargs):\n        """Stage-1 init from pretrained HF parts (SigLIP + BitNet bf16 master weights)."""\n        from transformers import AutoConfig, AutoTokenizer, BitNetForCausalLM, SiglipVisionModel\n\n        vt = SiglipVisionModel.from_pretrained(vision_model_name, vision_use_head=False, dtype=dtype)\n        lcfg = _strip_quant_config(AutoConfig.from_pretrained(llm_model_name))\n        llm = BitNetForCausalLM.from_pretrained(llm_model_name, config=lcfg, dtype=dtype)\n        tok = AutoTokenizer.from_pretrained(llm_model_name)\n        cfg = BitNetVLConfig(vision_model_name=vision_model_name, llm_model_name=llm_model_name, **cfg_kwargs)\n        return cls(cfg, vt, llm, tok)\n\n    @classmethod\n    def from_pretrained(cls, ckpt_dir: str | os.PathLike, dtype=None):\n        from transformers import AutoConfig, AutoTokenizer, BitNetForCausalLM, SiglipVisionConfig, SiglipVisionModel\n\n        ckpt_dir = Path(ckpt_dir)\n        cfg = BitNetVLConfig.load(ckpt_dir / "config.json")\n        vcfg = SiglipVisionConfig.from_dict(dict(cfg.vision_config))\n        vcfg.vision_use_head = False\n        vt = SiglipVisionModel(vcfg)\n        sd = load_file(str(ckpt_dir / "vision_tower.safetensors"))\n        vt.load_state_dict(sd, strict=True)\n        lm_dir = ckpt_dir / "language_model"\n        lcfg = _strip_quant_config(AutoConfig.from_pretrained(lm_dir))\n        llm = BitNetForCausalLM.from_pretrained(lm_dir, config=lcfg, dtype=dtype or torch.float32)\n        tok = AutoTokenizer.from_pretrained(lm_dir)\n        vision_ternary = cfg.vision_ternary\n        cfg.vision_ternary = False\n        model = cls(cfg, vt, llm, tok)\n        model.projector.load_state_dict(load_file(str(ckpt_dir / "projector.safetensors")), strict=True)\n        if vision_ternary:\n            model.ternarize_vision_()\n        if dtype is not None:\n            model.to(dtype)\n        return model\n\n    def save_pretrained(self, out_dir: str | os.PathLike, llm_dtype: torch.dtype | None = None):\n        out_dir = Path(out_dir)\n        out_dir.mkdir(parents=True, exist_ok=True)\n        self.config.save(out_dir / "config.json")\n        vsd = {k: v.detach().contiguous().cpu() for k, v in self.vision_tower.state_dict().items()}\n        save_file(vsd, str(out_dir / "vision_tower.safetensors"), metadata={"format": "pt"})\n        psd = {k: v.detach().contiguous().cpu() for k, v in self.projector.state_dict().items()}\n        save_file(psd, str(out_dir / "projector.safetensors"), metadata={"format": "pt"})\n        lm_dir = out_dir / "language_model"\n        llm = self.language_model\n        if llm_dtype is not None and llm.dtype != llm_dtype:\n            llm = copy.deepcopy(llm).to(llm_dtype)\n        llm.save_pretrained(lm_dir)\n        if self.tokenizer is not None:\n            self.tokenizer.save_pretrained(lm_dir)\n        _ensure_rope_theta(lm_dir)\n\n    # ---------------------------------------------------------------- quantization / freezing\n    def quantize_llm_(self) -> list[str]:\n        """Make every decoder linear (q/k/v/o/gate/up/down) a BitLinear for QAT. lm_head / embeddings stay FP.\n\n        transformers\' `BitNetForCausalLM` (v5) uses plain nn.Linear and only quantizes when a quantization_config\n        makes it swap in AutoBitLinear; we strip that config at load time, so there is no double quantization."""\n        self.config.llm_qat = True\n        return replace_linears(self.language_model, lambda n, m: bool(LLM_LINEAR_RE.search(n)))\n\n    def ternarize_vision_(self) -> list[str]:\n        """Swap SigLIP q/k/v/out_proj/fc1/fc2 for BitLinear. Patch embed, pos embed and LayerNorms stay FP."""\n        self.config.vision_ternary = True\n        return replace_linears(self.vision_tower, lambda n, m: bool(VISION_LINEAR_RE.search(n)))\n\n    def freeze_(self, vision: bool, projector: bool, llm: bool):\n        """Arguments say whether each part is TRAINABLE."""\n        self.vision_tower.requires_grad_(vision)\n        self.projector.requires_grad_(projector)\n        self.language_model.requires_grad_(llm)\n\n    def configure_stage(self, stage: int) -> dict[str, list[nn.Parameter]]:\n        """Set requires_grad for a recipe stage and return the trainable parameters by group."""\n        if stage == 1:\n            self.freeze_(vision=False, projector=True, llm=False)\n        elif stage == 2:\n            self.freeze_(vision=False, projector=True, llm=True)\n        elif stage == 3:\n            if not self.config.vision_ternary:\n                self.ternarize_vision_()\n            self.freeze_(vision=True, projector=False, llm=False)\n        else:\n            raise ValueError(stage)\n        groups = {\n            "vision": [p for p in self.vision_tower.parameters() if p.requires_grad],\n            "projector": [p for p in self.projector.parameters() if p.requires_grad],\n            "llm": [p for p in self.language_model.parameters() if p.requires_grad],\n        }\n        return {k: v for k, v in groups.items() if v}\n\n    def gradient_checkpointing_enable(self):\n        kw = {"gradient_checkpointing_kwargs": {"use_reentrant": False}}\n        self.language_model.gradient_checkpointing_enable(**kw)\n        self.vision_tower.gradient_checkpointing_enable(**kw)\n        self.language_model.config.use_cache = False\n\n    # ---------------------------------------------------------------- forward\n    @property\n    def device(self):\n        return self.projector.fc1.weight.device\n\n    def encode_images(self, pixel_values: torch.Tensor, output_hidden_states: bool = False):\n        vdtype = self.vision_tower.vision_model.embeddings.patch_embedding.weight.dtype\n        out = self.vision_tower(pixel_values=pixel_values.to(vdtype), output_hidden_states=output_hidden_states)\n        feats = pixel_shuffle(out.last_hidden_state, self.config.pixel_shuffle)\n        feats = self.projector(feats.to(self.projector.fc1.weight.dtype))\n        hs = tuple(out.hidden_states[1:]) if output_hidden_states else None\n        return feats, hs\n\n    def merge_inputs(self, input_ids, image_features=None, attention_mask=None, labels=None):\n        """Replace each sample\'s single `<image>` token by the image\'s N projected embeddings.\n\n        The i-th sample that contains an image token uses image_features[i\'] where i\' counts image-bearing samples.\n        Returns (inputs_embeds, attention_mask, labels), right-padded to the longest expanded row."""\n        emb = self.language_model.get_input_embeddings()\n        tid = self.config.image_token_id\n        bsz = input_ids.shape[0]\n        if attention_mask is None:\n            attention_mask = torch.ones_like(input_ids)\n        tok_embeds = emb(input_ids)\n        rows_e, rows_m, rows_l = [], [], []\n        img_i = 0\n        for b in range(bsz):\n            pos = (input_ids[b] == tid).nonzero(as_tuple=True)[0]\n            e, m = tok_embeds[b], attention_mask[b]\n            lab = labels[b] if labels is not None else None\n            if len(pos) == 0:\n                pass\n            elif len(pos) == 1:\n                if image_features is None or img_i >= image_features.shape[0]:\n                    raise ValueError("more <image> tokens than images")\n                p = int(pos[0])\n                f = image_features[img_i].to(e.dtype)\n                img_i += 1\n                n = f.shape[0]\n                e = torch.cat([e[:p], f, e[p + 1:]], 0)\n                m = torch.cat([m[:p], m.new_ones(n), m[p + 1:]], 0)\n                if lab is not None:\n                    lab = torch.cat([lab[:p], lab.new_full((n,), -100), lab[p + 1:]], 0)\n            else:\n                raise ValueError(f"sample {b} has {len(pos)} <image> tokens; exactly one is supported")\n            rows_e.append(e)\n            rows_m.append(m)\n            rows_l.append(lab)\n        if image_features is not None and img_i != image_features.shape[0]:\n            raise ValueError(f"{image_features.shape[0]} images but {img_i} <image> tokens")\n        L = max(r.shape[0] for r in rows_e)\n        out_e = tok_embeds.new_zeros(bsz, L, tok_embeds.shape[-1])\n        out_m = attention_mask.new_zeros(bsz, L)\n        out_l = None if labels is None else labels.new_full((bsz, L), -100)\n        for b in range(bsz):\n            n = rows_e[b].shape[0]\n            out_e[b, :n] = rows_e[b]\n            out_m[b, :n] = rows_m[b]\n            if out_l is not None:\n                out_l[b, :n] = rows_l[b]\n        return out_e, out_m, out_l\n\n    def forward(self, input_ids, pixel_values=None, attention_mask=None, labels=None,\n                output_vision_hidden_states: bool = False, **kwargs) -> BitNetVLOutput:\n        feats, vhs = (None, None)\n        if pixel_values is not None:\n            feats, vhs = self.encode_images(pixel_values, output_hidden_states=output_vision_hidden_states)\n        embeds, mask, labels = self.merge_inputs(input_ids, feats, attention_mask, labels)\n        out = self.language_model(inputs_embeds=embeds, attention_mask=mask, labels=labels, use_cache=False, **kwargs)\n        return BitNetVLOutput(loss=out.loss, logits=out.logits, vision_hidden_states=vhs, labels=labels)\n\n    # ---------------------------------------------------------------- inference\n    @torch.no_grad()\n    def generate(self, input_ids, pixel_values=None, attention_mask=None, max_new_tokens: int = 64,\n                 eos_token_ids=None) -> torch.Tensor:\n        """Greedy decoding with KV cache. Returns generated token ids (B, T) without the prompt."""\n        if eos_token_ids is None:\n            eos_token_ids = set()\n            if self.tokenizer is not None:\n                for t in (EOT_TOKEN, self.tokenizer.eos_token):\n                    if t is not None and t in self.tokenizer.get_vocab():\n                        eos_token_ids.add(self.tokenizer.convert_tokens_to_ids(t))\n        feats = self.encode_images(pixel_values)[0] if pixel_values is not None else None\n        embeds, mask, _ = self.merge_inputs(input_ids, feats, attention_mask)\n        out = self.language_model(inputs_embeds=embeds, attention_mask=mask, use_cache=True)\n        past = out.past_key_values\n        bsz = input_ids.shape[0]\n        done = torch.zeros(bsz, dtype=torch.bool, device=input_ids.device)\n        generated = []\n        nxt = out.logits[:, -1].argmax(-1)\n        for _ in range(max_new_tokens):\n            generated.append(nxt)\n            done |= torch.isin(nxt, torch.tensor(sorted(eos_token_ids), device=nxt.device, dtype=nxt.dtype)) \\\n                if eos_token_ids else torch.zeros_like(done)\n            if bool(done.all()):\n                break\n            mask = torch.cat([mask, mask.new_ones(bsz, 1)], 1)\n            out = self.language_model(input_ids=nxt[:, None], attention_mask=mask, past_key_values=past, use_cache=True)\n            past = out.past_key_values\n            nxt = out.logits[:, -1].argmax(-1)\n        return torch.stack(generated, 1)\n\n    @torch.no_grad()\n    def answer(self, image, question: str, max_new_tokens: int = 64) -> str:\n        """image: PIL image, path, or a preprocessed (3, H, W) tensor."""\n        from .data import build_prompt, preprocess_image\n\n        assert self.tokenizer is not None\n        pv = image if torch.is_tensor(image) else preprocess_image(image, self.config.image_size,\n                                                                  self.config.image_mean, self.config.image_std)\n        ids = self.tokenizer(build_prompt(question), add_special_tokens=False, return_tensors="pt").input_ids\n        gen = self.generate(ids.to(self.device), pv[None].to(self.device), max_new_tokens=max_new_tokens)\n        return self.tokenizer.decode(gen[0], skip_special_tokens=True).strip()\n\n\ndef bitlinear_names(module: nn.Module) -> list[str]:\n    return [n for n, m in module.named_modules() if isinstance(m, BitLinear)]\n',
    'bitnet_vl/tokenizer_utils.py': '"""Offline tokenizer helpers.\n\nThe real BitNet b1.58 2B4T tokenizer is the Llama-3 BPE tokenizer (128256 tokens). For offline / tiny tests we\nrebuild an equivalent HF fast tokenizer from the vocab GGUF shipped with llama.cpp\n(`3rdparty/llama.cpp/models/ggml-vocab-llama-bpe.gguf`): tokens, merges and token types come from the GGUF, the\npre-tokenizer / post-processor are the ones of Meta-Llama-3\'s `tokenizer.json`. The result hashes to "llama-bpe" in\n`utils/convert-hf-to-gguf-bitnet.py::get_vocab_base_pre`, so a checkpoint saved with it converts with the stock\nconverter.\n"""\n\nfrom __future__ import annotations\n\nimport sys\nfrom pathlib import Path\n\nIMAGE_TOKEN = "<image>"\nBOS_TOKEN = "<|begin_of_text|>"\nEOT_TOKEN = "<|eot_id|>"\n\nREPO_ROOT = Path(__file__).resolve().parents[2]\nDEFAULT_VOCAB_GGUF = REPO_ROOT / "3rdparty" / "llama.cpp" / "models" / "ggml-vocab-llama-bpe.gguf"\n\n# Renders exactly the training format of data.py (BitNet 2B4T style "Role: content<|eot_id|>").\nCHAT_TEMPLATE = (\n    "{{ bos_token }}{% for message in messages %}"\n    "{{ (message[\'role\'] | capitalize) + \': \' + (message[\'content\'] | trim) + \'<|eot_id|>\' }}"\n    "{% endfor %}{% if add_generation_prompt %}{{ \'Assistant: \' }}{% endif %}"\n)\n\nLLAMA3_PRETOKENIZE_REGEX = (\n    r"(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"\n)\n\n\ndef _read_gguf_vocab(path: Path):\n    gguf_py = REPO_ROOT / "3rdparty" / "llama.cpp" / "gguf-py"\n    if str(gguf_py) not in sys.path:\n        sys.path.insert(0, str(gguf_py))\n    from gguf import GGUFReader  # noqa: E402\n\n    r = GGUFReader(str(path))\n\n    def strarr(key):\n        f = r.fields[key]\n        return [bytes(f.parts[i]).decode("utf-8") for i in f.data]\n\n    tokens = strarr("tokenizer.ggml.tokens")\n    merges = strarr("tokenizer.ggml.merges")\n    tt = r.fields["tokenizer.ggml.token_type"]\n    types = [int(tt.parts[i][0]) for i in tt.data]\n    return tokens, merges, types\n\n\ndef build_llama3_tokenizer(vocab_gguf: str | Path = DEFAULT_VOCAB_GGUF):\n    """Build a Llama-3 `PreTrainedTokenizerFast` from llama.cpp\'s vocab GGUF (no network)."""\n    from tokenizers import AddedToken, Regex, Tokenizer, decoders, models, pre_tokenizers, processors\n    from transformers import PreTrainedTokenizerFast\n\n    tokens, merges, types = _read_gguf_vocab(Path(vocab_gguf))\n    vocab = {t: i for i, t in enumerate(tokens) if types[i] == 1}\n    merge_pairs = [tuple(m.split(" ", 1)) for m in merges]\n    tok = Tokenizer(models.BPE(vocab=vocab, merges=merge_pairs, ignore_merges=True, fuse_unk=False, byte_fallback=False))\n    tok.pre_tokenizer = pre_tokenizers.Sequence(\n        [\n            pre_tokenizers.Split(Regex(LLAMA3_PRETOKENIZE_REGEX), behavior="isolated", invert=False),\n            pre_tokenizers.ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=False),\n        ]\n    )\n    tok.decoder = decoders.ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)\n    # special (control) tokens keep their GGUF ids: add them in id order\n    specials = [(i, t) for i, t in enumerate(tokens) if types[i] != 1]\n    assert [i for i, _ in specials] == list(range(len(vocab), len(tokens))), "special tokens must be contiguous"\n    tok.add_special_tokens([AddedToken(t, special=True, normalized=False) for _, t in specials])\n    bos_id = tokens.index(BOS_TOKEN)\n    tok.post_processor = processors.Sequence(\n        [\n            processors.ByteLevel(trim_offsets=False),\n            processors.TemplateProcessing(\n                single=f"{BOS_TOKEN} $A", pair=f"{BOS_TOKEN} $A {BOS_TOKEN} $B", special_tokens=[(BOS_TOKEN, bos_id)]\n            ),\n        ]\n    )\n    hf = PreTrainedTokenizerFast(\n        tokenizer_object=tok,\n        bos_token=BOS_TOKEN,\n        eos_token=EOT_TOKEN,\n        pad_token=EOT_TOKEN,\n        clean_up_tokenization_spaces=True,\n        model_max_length=4096,\n    )\n    hf.chat_template = CHAT_TEMPLATE\n    return hf\n\n\ndef add_image_token(tokenizer) -> int:\n    """Add the `<image>` special token (idempotent) and return its id."""\n    if IMAGE_TOKEN not in tokenizer.get_vocab():\n        tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = EOT_TOKEN if EOT_TOKEN in tokenizer.get_vocab() else tokenizer.eos_token\n    return tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)\n',
    'bitnet_vl/make_tiny_ckpt.py': '"""Build a tiny randomly-initialized BitNet-VL checkpoint in the contract layout (for exporter / e2e tests).\n\n    cd multimodal && python -m bitnet_vl.make_tiny_ckpt OUT_DIR [--vision-ternary] [--seed 0]\n\nShapes (all linear K dims are multiples of 128 so every ternary matrix can be stored I2_S):\n  vision: SigLIP hidden 128, mlp 512, 2 layers, 2 heads, image 64, patch 16 -> 4x4 patches -> 2x2 shuffle -> 4 tokens\n  projector: LN(512) -> 512x256 -> GELU -> 256x256\n  LLM: BitNet hidden 256, intermediate 512, 2 layers, 4 heads (head_dim 64), 2 kv heads, relu2, tied embeddings,\n       Llama-3 tokenizer rebuilt offline from 3rdparty/llama.cpp/models/ggml-vocab-llama-bpe.gguf + `<image>`\n       (vocab 128257). `language_model/` converts with `utils/convert-hf-to-gguf-bitnet.py --outtype i2_s`.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\n\nimport torch\n\nfrom .modeling import BitNetVLForConditionalGeneration\nfrom .tokenizer_utils import build_llama3_tokenizer\n\n\ndef tiny_vision_config(hidden=128, intermediate=512, layers=2, heads=2, image_size=64, patch=16):\n    from transformers import SiglipVisionConfig\n\n    return SiglipVisionConfig(hidden_size=hidden, intermediate_size=intermediate, num_hidden_layers=layers,\n                              num_attention_heads=heads, image_size=image_size, patch_size=patch,\n                              vision_use_head=False)\n\n\ndef tiny_llm_config(vocab_size=128256, hidden=256, intermediate=512, layers=2, heads=4, kv_heads=2):\n    from transformers import BitNetConfig\n\n    return BitNetConfig(vocab_size=vocab_size, hidden_size=hidden, intermediate_size=intermediate,\n                        num_hidden_layers=layers, num_attention_heads=heads, num_key_value_heads=kv_heads,\n                        max_position_embeddings=2048, tie_word_embeddings=True, bos_token_id=128000,\n                        eos_token_id=128009, pad_token_id=128009)\n\n\ndef build_tiny_model(seed: int = 0, vision_ternary: bool = False, tokenizer=None, llm_hidden=256, vision_hidden=128):\n    torch.manual_seed(seed)\n    tok = tokenizer if tokenizer is not None else build_llama3_tokenizer()\n    vc = tiny_vision_config(hidden=vision_hidden)\n    lc = tiny_llm_config(vocab_size=len(tok), hidden=llm_hidden)\n    model = BitNetVLForConditionalGeneration.from_configs(\n        vc, lc, tok, vision_model_name="tiny-random-siglip", llm_model_name="tiny-random-bitnet",\n        vision_ternary=vision_ternary)\n    # random init leaves LayerNorm/bias at trivial values; perturb so exporters are tested on non-trivial tensors\n    with torch.no_grad():\n        for n, p in model.named_parameters():\n            if p.ndim == 1 and ("norm" in n or "bias" in n):\n                p.add_(0.02 * torch.randn_like(p))\n    return model.eval()\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("out_dir")\n    ap.add_argument("--vision-ternary", action="store_true", help="mark the vision tower ternary (BitLinear)")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--llm-dtype", choices=["bf16", "f32"], default="bf16")\n    args = ap.parse_args(argv)\n    model = build_tiny_model(args.seed, args.vision_ternary)\n    model.save_pretrained(args.out_dir, llm_dtype=torch.bfloat16 if args.llm_dtype == "bf16" else None)\n    c = model.config\n    print(f"saved tiny BitNet-VL checkpoint to {args.out_dir}: image {c.image_size}px, {c.num_image_tokens} image "\n          f"tokens, vision d={c.vision_hidden}, llm d={c.llm_hidden}, vocab {len(model.tokenizer)}, "\n          f"image_token_id {c.image_token_id}, vision_ternary={c.vision_ternary}")\n\n\nif __name__ == "__main__":\n    main()\n',
    'bitnet_vl/latent/__init__.py': '"""Latent-reasoning experiment for BitNet (Coconut-style compressed feedback vs pause tokens vs CoT).\n\nSee multimodal/README.md, section "Latent reasoning". Modules:\n  tasks.py       synthetic variable-tracking programs\n  compressor.py  Perceiver-style LatentCompressor (layer-i hidden states -> K soft tokens)\n  lora.py        minimal LoRA over nn.Linear / BitLinear\n  reasoner.py    LatentReasoner (modes none / latent / pause / cot), KV-cache reuse, curriculum\n  train_latent.py CLI + run(config)\n"""\n',
    'bitnet_vl/latent/tasks.py': '"""Synthetic variable-tracking programs.\n\nExample (hops=3, distractors=2)::\n\n    q = 7\n    x = 4\n    m = q + 5\n    t = m * 3\n    k = x - 1\n    p = t - q\n    print(p)\n\n* The *chain* is ``root = c; v1 = root op o1; v2 = v1 op o2; ...``; ``hops`` = number of computed chain assignments\n  (= number of trace steps). Operands are a constant 1..9 or (sometimes) an earlier chain variable.\n* *Distractors* are assignments not on the chain (constants or expressions of earlier variables), interleaved at\n  random positions (always after the variables they use).\n* All intermediate values stay in [0, 99] (operations that would leave the range are re-sampled), so the program is\n  plain Python and answers have at most 2 digits; ``exec`` of the program gives the same values (tested).\n* ``trace_steps`` are the evaluated chain assignments, e.g. ``["m = 12", "t = 36", "p = 29"]``.\n\nEverything is deterministic given the seed. `TaskSampler` streams training examples; `make_test_set` builds a\nfixed test set from a disjoint seed space.\n"""\n\nfrom __future__ import annotations\n\nimport random\nimport string\nfrom dataclasses import dataclass\n\nOPS = ("+", "-", "*")\nMAX_VAL = 99\nTEST_SEED_OFFSET = 1_000_003  # test streams use seeds disjoint from the training stream\n\n\ndef _apply(a: int, op: str, b: int) -> int:\n    return a + b if op == "+" else a - b if op == "-" else a * b\n\n\ndef _sample_op(rng: random.Random, value: int, operands: list[tuple[str, int]]) -> tuple[str, str, int]:\n    """Pick (op, operand_text, result) with result in [0, MAX_VAL]. operands: available (name, value) vars."""\n    for _ in range(50):\n        op = rng.choice(OPS)\n        if operands and rng.random() < 0.3:\n            name, val = rng.choice(operands)\n        else:\n            val = rng.randint(1, 9) if op != "*" else rng.randint(2, 5)\n            name = str(val)\n        r = _apply(value, op, val)\n        if 0 <= r <= MAX_VAL:\n            return op, name, r\n    return ("+", "1", value + 1) if value < MAX_VAL else ("-", "1", value - 1)\n\n\ndef make_example(rng: random.Random, hops: int, distractors: int = 0) -> dict:\n    """One program. Returns dict(prompt, answer, trace_steps, hops, program_lines, query)."""\n    if hops < 1:\n        raise ValueError("hops >= 1")\n    n_vars = hops + 1 + distractors\n    if n_vars > 26:\n        raise ValueError("too many variables for single-letter names")\n    names = list(string.ascii_lowercase)\n    rng.shuffle(names)\n    chain_names, dist_names = names[:hops + 1], names[hops + 1:n_vars]\n\n    # chain\n    root_val = rng.randint(0, 20)\n    chain_lines = [f"{chain_names[0]} = {root_val}"]\n    values = {chain_names[0]: root_val}\n    trace = []\n    cur = root_val\n    for k in range(1, hops + 1):\n        prev = chain_names[k - 1]\n        earlier = [(n, values[n]) for n in chain_names[:k - 1]]\n        op, operand, cur = _sample_op(rng, cur, earlier)\n        line = f"{chain_names[k]} = {prev} {op} {operand}"\n        values[chain_names[k]] = cur\n        chain_lines.append(line)\n        trace.append(f"{chain_names[k]} = {cur}")\n\n    # interleave distractors: choose insertion slots among the chain lines (slot i = before chain line i, or end)\n    lines: list[str] = list(chain_lines)\n    defined_at = {n: i for i, n in enumerate(chain_names)}  # index in `lines` of definition\n    dvals: dict[str, int] = {}\n    for d in dist_names:\n        pos = rng.randint(0, len(lines))  # insert before lines[pos]\n        avail = [(n, values.get(n, dvals.get(n))) for n, i in defined_at.items() if i < pos]\n        if avail and rng.random() < 0.6:\n            src, sval = rng.choice(avail)\n            others = [a for a in avail if a[0] != src]\n            op, operand, val = _sample_op(rng, sval, others)\n            line = f"{d} = {src} {op} {operand}"\n        else:\n            val = rng.randint(0, 20)\n            line = f"{d} = {val}"\n        lines.insert(pos, line)\n        for n in defined_at:\n            if defined_at[n] >= pos:\n                defined_at[n] += 1\n        defined_at[d] = pos\n        dvals[d] = val\n    query = chain_names[-1]\n    lines.append(f"print({query})")\n    prompt = "\\n".join(lines) + "\\n"\n    return dict(prompt=prompt, answer=values[query], trace_steps=trace, hops=hops, program_lines=lines, query=query)\n\n\ndef exec_program(prompt: str) -> int:\n    """Reference answer by executing the program with print captured."""\n    import contextlib\n    import io\n\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        exec(prompt, {"__builtins__": {"print": print}}, {})  # noqa: S102 (trusted generated code)\n    return int(buf.getvalue().strip())\n\n\ndef trace_values(prompt: str, trace_steps: list[str]) -> bool:\n    """Check that each trace line \'v = n\' matches the value of v after executing the program."""\n    env: dict = {}\n    exec(prompt.replace("print(", "_q = ("), {"__builtins__": {}}, env)  # noqa: S102\n    for t in trace_steps:\n        name, val = (s.strip() for s in t.split("="))\n        if env[name] != int(val):\n            return False\n    return True\n\n\ndef parse_hops(spec: str | list | tuple) -> list[int]:\n    """\'2-6\' -> [2..6]; \'2,4,8\' -> [2,4,8]; list passthrough."""\n    if isinstance(spec, (list, tuple)):\n        return [int(x) for x in spec]\n    spec = str(spec).strip()\n    out: list[int] = []\n    for part in spec.split(","):\n        if "-" in part:\n            a, b = part.split("-")\n            out.extend(range(int(a), int(b) + 1))\n        elif part:\n            out.append(int(part))\n    return out\n\n\n@dataclass\nclass TaskSampler:\n    """Infinite deterministic stream of training examples with hops uniform over `hops`."""\n\n    hops: list[int]\n    distractors: int = 0\n    seed: int = 0\n\n    def __post_init__(self):\n        self.rng = random.Random(self.seed)\n\n    def sample(self) -> dict:\n        return make_example(self.rng, self.rng.choice(self.hops), self.distractors)\n\n    def batch(self, n: int) -> list[dict]:\n        return [self.sample() for _ in range(n)]\n\n\ndef make_test_set(hops: list[int], n_per_hops: int, distractors: int = 0, seed: int = 0) -> dict[int, list[dict]]:\n    """Fixed test examples per hop count, from a seed space disjoint from the training stream."""\n    out = {}\n    for h in hops:\n        rng = random.Random(TEST_SEED_OFFSET + 7919 * h + seed)\n        out[h] = [make_example(rng, h, distractors) for _ in range(n_per_hops)]\n    return out\n',
    'bitnet_vl/latent/compressor.py': '"""LatentCompressor: K learned queries cross-attend (Perceiver-style, pre-LN) over the layer-i hidden states of all\npositions seen so far, then Linear -> RMSNorm rescaled to the RMS of the LLM\'s input-embedding rows.\n\nThe output rescaling matters for BitNet: every BitLinear quantizes its input per token with absmax int8, and the\nfirst op on an input embedding is an RMSNorm, but the residual stream keeps the raw embedding, so soft tokens whose\nnorm is far from that of real token embeddings would be out of distribution for every later layer. The compressor\nis always full precision (never BitLinear).\n"""\n\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef embedding_rms(emb_weight: torch.Tensor) -> float:\n    """Mean over rows of the per-row RMS of an embedding table (the typical norm/sqrt(d) of an input token)."""\n    w = emb_weight.detach()\n    tot = 0.0\n    for i in range(0, w.shape[0], 16384):  # chunked: a 128k x 2560 table in fp32 would need 1.3 GB at once\n        tot += float(w[i:i + 16384].float().pow(2).mean(-1).sqrt().sum())\n    return tot / w.shape[0]\n\n\ndef rms_rescale(x: torch.Tensor, target_rms: float | torch.Tensor, eps: float = 1e-6) -> torch.Tensor:\n    xf = x.float()\n    return xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps) * target_rms\n\n\nclass CrossAttentionBlock(nn.Module):\n    def __init__(self, d: int, n_heads: int, mlp_ratio: float = 2.0):\n        super().__init__()\n        assert d % n_heads == 0\n        self.h = n_heads\n        self.ln_q = nn.LayerNorm(d)\n        self.ln_kv = nn.LayerNorm(d)\n        self.q = nn.Linear(d, d)\n        self.kv = nn.Linear(d, 2 * d)\n        self.o = nn.Linear(d, d)\n        self.ln_mlp = nn.LayerNorm(d)\n        hidden = int(d * mlp_ratio)\n        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Linear(hidden, d))\n\n    def forward(self, x, mem, mem_mask=None):\n        """x: (B, K, d) latents; mem: (B, L, d); mem_mask: (B, L) 1 = valid."""\n        b, k, d = x.shape\n        q = self.q(self.ln_q(x)).view(b, k, self.h, d // self.h).transpose(1, 2)\n        kv = self.kv(self.ln_kv(mem)).view(b, mem.shape[1], 2, self.h, d // self.h)\n        key, val = kv.unbind(2)\n        key, val = key.transpose(1, 2), val.transpose(1, 2)\n        attn_mask = None\n        if mem_mask is not None:\n            attn_mask = mem_mask.bool()[:, None, None, :]  # (B, 1, 1, L)\n        a = F.scaled_dot_product_attention(q, key, val, attn_mask=attn_mask)\n        x = x + self.o(a.transpose(1, 2).reshape(b, k, d))\n        return x + self.mlp(self.ln_mlp(x))\n\n\nclass LatentCompressor(nn.Module):\n    """``forward(memory, memory_mask, step) -> (B, K, d_model)`` soft tokens with per-token RMS == target_rms.\n\n    Args:\n        d_model: LLM hidden size (memory and output dim).\n        K: number of soft tokens per latent step.\n        n_layers / n_heads: cross-attention blocks over the memory.\n        target_rms: RMS of the LLM input embedding rows (`embedding_rms(lm.get_input_embeddings().weight)`).\n        max_steps: number of learned step embeddings added to the queries (step index is clipped), so the compressor\n            knows which reasoning step it is producing; the memory alone already grows with every step.\n    """\n\n    def __init__(self, d_model: int, K: int, n_layers: int = 1, n_heads: int = 8, target_rms: float = 1.0,\n                 max_steps: int = 16, mlp_ratio: float = 2.0):\n        super().__init__()\n        self.d_model, self.K, self.max_steps = d_model, K, max_steps\n        self.queries = nn.Parameter(torch.randn(K, d_model) / math.sqrt(d_model))\n        self.step_emb = nn.Parameter(torch.zeros(max_steps, d_model))\n        self.blocks = nn.ModuleList([CrossAttentionBlock(d_model, n_heads, mlp_ratio) for _ in range(n_layers)])\n        self.ln_out = nn.LayerNorm(d_model)\n        self.out = nn.Linear(d_model, d_model)\n        self.register_buffer("target_rms", torch.tensor(float(target_rms)), persistent=True)\n\n    def forward(self, memory: torch.Tensor, memory_mask: torch.Tensor | None = None, step: int = 0) -> torch.Tensor:\n        b = memory.shape[0]\n        s = min(int(step), self.max_steps - 1)\n        x = (self.queries + self.step_emb[s]).unsqueeze(0).expand(b, -1, -1)\n        mem = memory if torch.is_autocast_enabled(memory.device.type) else memory.to(self.queries.dtype)\n        for blk in self.blocks:\n            x = blk(x, mem, memory_mask)\n        y = self.out(self.ln_out(x))\n        return rms_rescale(y, self.target_rms)\n\n\nclass PauseTokens(nn.Module):\n    """Learned constant pause embeddings (K per step, shared across steps), RMS-matched like the compressor output.\n    No dependence on hidden states: the control for the extra compute of latent steps."""\n\n    def __init__(self, d_model: int, K: int, target_rms: float = 1.0, init: torch.Tensor | None = None):\n        super().__init__()\n        self.K = K\n        w = init.detach().float().clone() if init is not None else torch.randn(K, d_model)\n        self.emb = nn.Parameter(w)\n        self.register_buffer("target_rms", torch.tensor(float(target_rms)), persistent=True)\n\n    def forward(self, batch_size: int, R: int) -> torch.Tensor:\n        e = rms_rescale(self.emb, self.target_rms)  # (K, d)\n        return e.unsqueeze(0).expand(batch_size, -1, -1).repeat(1, R, 1)  # (B, R*K, d)\n',
    'bitnet_vl/latent/lora.py': '"""Minimal LoRA (no peft) for nn.Linear / BitLinear.\n\n``y = base(x) + (dropout(x) @ A^T @ B^T) * alpha / r``. The LoRA path is full precision and is added to the\n(BitLinear W1.58A8) base output; the base weight is frozen. B is zero-initialized so a freshly wrapped model is\nexactly the base model.\n\nFor a frozen BitLinear base, `LoRALinear.prequantize_base_()` replaces the latent weight by its ternary fake-quant\nonce (in place, no extra memory) so the forward skips the per-call absmean quantization. This gives the same output\nas the BitLinear forward (tested) but the module can no longer be trained or exported as master weights, so it is\nonly used when the base is frozen.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Callable\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom ..bitlinear import BitLinear, activation_quant, weight_quant\n\nDEFAULT_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")\nLORA_KEY = "lora_"\n\n\nclass LoRALinear(nn.Module):\n    def __init__(self, base: nn.Linear, r: int = 16, alpha: float = 32.0, dropout: float = 0.0):\n        super().__init__()\n        self.base = base\n        self.r, self.alpha = r, alpha\n        self.scaling = alpha / r\n        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, dtype=torch.float32, device=base.weight.device))\n        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, dtype=torch.float32, device=base.weight.device))\n        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))\n        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n        self.prequantized = False\n        base.weight.requires_grad_(False)\n        if base.bias is not None:\n            base.bias.requires_grad_(False)\n\n    @property\n    def in_features(self):\n        return self.base.in_features\n\n    @property\n    def out_features(self):\n        return self.base.out_features\n\n    @property\n    def weight(self):  # some HF utilities peek at .weight (dtype/device)\n        return self.base.weight\n\n    @torch.no_grad()\n    def prequantize_base_(self):\n        if isinstance(self.base, BitLinear) and not self.prequantized:\n            self.base.weight.data.copy_(weight_quant(self.base.weight.data))\n            self.prequantized = True\n\n    def base_forward(self, x):\n        if self.prequantized:\n            if self.base.quantize_activations:\n                x = x + (activation_quant(x) - x).detach()  # STE so grads still reach earlier layers\' LoRA\n            return F.linear(x, self.base.weight, self.base.bias)\n        return self.base(x)\n\n    def forward(self, x):\n        y = self.base_forward(x)\n        a = self.lora_dropout(x)\n        if not torch.is_autocast_enabled(x.device.type):\n            a = a.to(self.lora_A.dtype)\n        lo = F.linear(F.linear(a, self.lora_A), self.lora_B) * self.scaling\n        return y + lo.to(y.dtype)\n\n    def extra_repr(self):\n        return f"r={self.r}, alpha={self.alpha}, prequantized={self.prequantized}"\n\n\ndef _default_predicate(targets):\n    pat = re.compile(r"(^|\\.)(" + "|".join(re.escape(t) for t in targets) + r")$")\n    return lambda name, mod: bool(pat.search(name))\n\n\ndef apply_lora(model: nn.Module, r: int = 16, alpha: float = 32.0, dropout: float = 0.0,\n               predicate: Callable[[str, nn.Module], bool] | None = None,\n               targets=DEFAULT_TARGETS, prequantize: bool = False) -> list[str]:\n    """Wrap every nn.Linear (incl. BitLinear) selected by `predicate(name, module)` (default: name ends with one of\n    `targets`) in a LoRALinear. Returns the wrapped names. Does NOT freeze the rest of the model (see `freeze_base`)."""\n    pred = predicate or _default_predicate(targets)\n    names = []\n    for name, sub in list(model.named_modules()):\n        if isinstance(sub, nn.Linear) and pred(name, sub):\n            w = LoRALinear(sub, r=r, alpha=alpha, dropout=dropout)\n            if prequantize:\n                w.prequantize_base_()\n            model.set_submodule(name, w)\n            names.append(name)\n    return names\n\n\ndef freeze_base(model: nn.Module):\n    """requires_grad=False for every parameter except LoRA A/B."""\n    for n, p in model.named_parameters():\n        p.requires_grad_(LORA_KEY in n.rsplit(".", 1)[-1])\n\n\ndef lora_parameters(model: nn.Module) -> list[nn.Parameter]:\n    return [p for n, p in model.named_parameters() if LORA_KEY in n.rsplit(".", 1)[-1]]\n\n\ndef count_params(module: nn.Module, trainable_only: bool = True) -> int:\n    seen, total = set(), 0\n    for p in module.parameters():\n        if id(p) in seen or (trainable_only and not p.requires_grad):\n            continue\n        seen.add(id(p))\n        total += p.numel()\n    return total\n\n\ndef lora_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:\n    return {k: v.detach().cpu() for k, v in model.state_dict().items() if LORA_KEY in k.rsplit(".", 1)[-1]}\n\n\ndef save_trainable(path: str | Path, lm: nn.Module, modules: dict[str, nn.Module] | None = None,\n                   include_base: bool = False):\n    """Save LoRA weights of `lm` plus the full state of each extra module (e.g. {"compressor": comp}).\n    include_base=True also saves the full LM state dict (used with --train-base)."""\n    sd = {"lm." + k: v for k, v in (lm.state_dict() if include_base else lora_state_dict(lm)).items()}\n    for prefix, m in (modules or {}).items():\n        if m is not None:\n            sd.update({f"{prefix}.{k}": v.detach().cpu() for k, v in m.state_dict().items()})\n    torch.save({k: v.detach().cpu().contiguous() for k, v in sd.items()}, str(path))\n\n\ndef load_trainable(path: str | Path, lm: nn.Module, modules: dict[str, nn.Module] | None = None, strict=True):\n    sd = torch.load(str(path), map_location="cpu", weights_only=True)\n    lm_sd = {k[3:]: v for k, v in sd.items() if k.startswith("lm.")}\n    missing, unexpected = lm.load_state_dict(lm_sd, strict=False)\n    if strict:\n        need = [k for k in lora_state_dict(lm) if k not in lm_sd]\n        if need or unexpected:\n            raise RuntimeError(f"LoRA keys missing {need[:5]} unexpected {list(unexpected)[:5]}")\n    for prefix, m in (modules or {}).items():\n        if m is None:\n            continue\n        sub = {k[len(prefix) + 1:]: v for k, v in sd.items() if k.startswith(prefix + ".")}\n        m.load_state_dict(sub, strict=strict)\n    return sd\n',
    'bitnet_vl/latent/reasoner.py': '"""LatentReasoner: prompt -> R reasoning steps (latent / pause) -> answer, on a HF BitNetForCausalLM.\n\nSequence layout (all sequences in a batch share one layout; prompts are LEFT-padded so every row ends at the same\ncache index):\n\n    [pad..pad BOS program\\\\n] [thought step 1: K positions] ... [step R] [remaining trace lines] Answer: 23<|eot_id|>\n     \\\\-------- prompt -------/ \\\\------------- R*K positions -------------/ \\\\---------- target (right-padded) -------/\n\nModes:\n  none    R = 0, target = "Answer: <a><eot>".\n  latent  each step t: soft = compressor(layer-i states of every position seen so far, step=t) (K vectors), fed via\n          inputs_embeds while reusing the KV cache (only the K new positions are computed); their layer-i states are\n          appended to the memory the compressor reads at step t+1.\n  pause   the same R*K extra positions, but the inputs are learned constant pause embeddings (no feedback).\n  cot     R = 0, target = all trace lines ("b = 5\\\\n...") + "Answer: <a><eot>", loss on all of it.\n\nLoss: CE on target tokens only (trace lines, answer digits, eot). The "Answer:" separator is in the loss only when\ntrace lines precede it (cot / non-final curriculum stages), because then the model must decide when to stop.\n\nCurriculum (Coconut): at stage s < S (S = R), the first min(s, n) trace steps are replaced by s thought steps and\nthe remaining trace stays explicit (with loss); at stage >= S all trace is latent with R thought steps.\n\nPosition ids / masks: transformers 5 BitNet uses ``position_ids = cache_position`` unless given, which is wrong for\nleft padding, so explicit position_ids = (number of real tokens before this one) are passed for every chunk, together\nwith a 2-D attention mask over (past + new) positions. cache_position (index into the cache) is left to the model\n(= arange(new) + cached length), which is right because the cache layout is shared by all rows.\n\nGradient checkpointing: HF\'s GradientCheckpointingLayer drops the KV cache when checkpointing, so we use our own\nper-decoder-layer checkpointing (`enable_checkpointing`) together with `LatentCache`, whose layers update\nidempotently (keys = cache[:start] ++ new, stored only the first time), so recomputation in backward neither\nduplicates entries nor reads later entries.\n"""\n\nfrom __future__ import annotations\n\nimport functools\nimport re\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom transformers.cache_utils import Cache, DynamicLayer\n\nMODES = ("none", "latent", "pause", "cot")\nSEP = "Answer:"\nIGNORE = -100\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# KV cache + checkpointing\n# ----------------------------------------------------------------------------------------------------------------\nclass CheckpointSafeLayer(DynamicLayer):\n    """DynamicLayer whose update is a pure function of (cache[:start], new) with start = cache_position[0]."""\n\n    def update(self, key_states, value_states, cache_kwargs=None):\n        if not self.is_initialized:\n            self.lazy_initialization(key_states, value_states)\n        cur = self.keys.shape[-2] if self.keys.numel() else 0\n        cp = (cache_kwargs or {}).get("cache_position")\n        start = int(cp[0]) if cp is not None and cp.numel() else cur\n        if start == cur:\n            if cur == 0:\n                self.keys, self.values = key_states, value_states\n            else:\n                self.keys = torch.cat([self.keys, key_states], dim=-2)\n                self.values = torch.cat([self.values, value_states], dim=-2)\n            return self.keys, self.values\n        if start > cur:\n            raise RuntimeError(f"cache gap: write at {start}, cache has {cur}")\n        # recomputation (gradient checkpointing): rebuild without touching the stored cache\n        return (torch.cat([self.keys[..., :start, :], key_states], dim=-2),\n                torch.cat([self.values[..., :start, :], value_states], dim=-2))\n\n\nclass LatentCache(Cache):\n    def __init__(self, num_layers: int):\n        super().__init__(layers=[CheckpointSafeLayer() for _ in range(num_layers)])\n\n\nclass _CkptCallMixin:\n    def __call__(self, *args, **kwargs):\n        if getattr(self, "_latent_ckpt", False) and self.training and torch.is_grad_enabled():\n            return checkpoint(functools.partial(nn.Module.__call__, self, **kwargs), *args, use_reentrant=False)\n        return nn.Module.__call__(self, *args, **kwargs)\n\n\ndef enable_checkpointing(lm, enabled: bool = True):\n    """Per-decoder-layer activation checkpointing that keeps the KV cache (see module doc)."""\n    for layer in lm.model.layers:\n        cls = type(layer)\n        if not issubclass(cls, _CkptCallMixin):\n            layer.__class__ = type(f"Ckpt{cls.__name__}", (_CkptCallMixin, cls), {})\n        layer._latent_ckpt = enabled\n        layer.gradient_checkpointing = False  # never use HF\'s (it drops past_key_values)\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# formatting / curriculum\n# ----------------------------------------------------------------------------------------------------------------\nclass Formatter:\n    def __init__(self, tokenizer, sep: str = SEP):\n        self.tok = tokenizer\n        vocab = tokenizer.get_vocab()\n        self.bos = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else vocab["<|begin_of_text|>"]\n        self.eot = vocab["<|eot_id|>"] if "<|eot_id|>" in vocab else tokenizer.eos_token_id\n        self.pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else self.eot\n        self.sep_text = sep\n        self.sep = self.enc(sep)\n\n    def enc(self, text: str) -> list[int]:\n        return self.tok(text, add_special_tokens=False).input_ids if text else []\n\n    def prompt(self, text: str) -> list[int]:\n        return [self.bos] + self.enc(text)\n\n    def trace(self, lines: list[str]) -> list[int]:\n        return self.enc("".join(line + "\\n" for line in lines))\n\n    def answer(self, ans) -> list[int]:\n        return self.enc(f" {ans}") + [self.eot]\n\n\ndef plan_example(ex: dict, mode: str, R: int, stage: int | None = None) -> tuple[int, list[str]]:\n    """-> (number of thought steps, explicit trace lines kept before the answer).\n\n    stage=None means "no curriculum" (final setting). For latent/pause with a curriculum, there are S+1 stages\n    0..S with S = R: stage s < S keeps trace[min(s, n):] and uses s thought steps; stage >= S is all-latent."""\n    trace = list(ex["trace_steps"])\n    if mode == "none":\n        return 0, []\n    if mode == "cot":\n        return 0, trace\n    if stage is None or stage >= R:\n        return R, []\n    return stage, trace[min(stage, len(trace)):]\n\n\ndef num_stages(R: int) -> int:\n    return R + 1\n\n\ndef _left_pad(seqs: list[list[int]], pad: int):\n    L = max(len(s) for s in seqs)\n    ids = torch.full((len(seqs), L), pad, dtype=torch.long)\n    mask = torch.zeros((len(seqs), L), dtype=torch.long)\n    for i, s in enumerate(seqs):\n        ids[i, L - len(s):] = torch.tensor(s, dtype=torch.long)\n        mask[i, L - len(s):] = 1\n    return ids, mask\n\n\ndef _right_pad(seqs: list[list[int]], pad: int, fill=None):\n    L = max(len(s) for s in seqs)\n    ids = torch.full((len(seqs), L), pad if fill is None else fill, dtype=torch.long)\n    mask = torch.zeros((len(seqs), L), dtype=torch.long)\n    for i, s in enumerate(seqs):\n        ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)\n        mask[i, :len(s)] = 1\n    return ids, mask\n\n\n_NUM_RE = re.compile(r"-?\\d+")\n\n\ndef parse_answer(text: str, forced_sep: bool) -> int | None:\n    if not forced_sep:\n        if SEP not in text:\n            return None\n        text = text.split(SEP, 1)[1]\n    m = _NUM_RE.search(text)\n    return int(m.group()) if m else None\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# reasoner\n# ----------------------------------------------------------------------------------------------------------------\nclass LatentReasoner(nn.Module):\n    """\n    Args:\n        lm: BitNetForCausalLM (with BitLinear / LoRA already applied).\n        compressor: LatentCompressor (mode \'latent\'); pause: PauseTokens (mode \'pause\').\n        read_layer: HF hidden_states index i in [1, num_layers]: output of decoder layer i-1 (i = num_layers is the\n            final-norm output, as in HF\'s hidden_states[-1]).\n        K: soft tokens per step. R: default thought steps. bptt: if > 0, when producing thought t the compressor\'s\n            memory entries from thoughts older than the last `bptt` steps are detached (truncated recurrence).\n    """\n\n    def __init__(self, lm, compressor=None, read_layer: int | None = None, K: int = 4, mode: str = "latent",\n                 tokenizer=None, pause=None, R: int = 4, bptt: int = 0, sep: str = SEP):\n        super().__init__()\n        assert mode in MODES, mode\n        self.lm, self.compressor, self.pause = lm, compressor, pause\n        self.mode, self.K, self.R, self.bptt = mode, K, R, bptt\n        n = lm.config.num_hidden_layers\n        self.num_layers = n\n        self.read_layer = read_layer if read_layer is not None else default_read_layer(n)\n        assert 1 <= self.read_layer <= n\n        if mode == "latent":\n            assert compressor is not None and compressor.K == K\n        if mode == "pause":\n            assert pause is not None and pause.K == K\n        self.fmt = Formatter(tokenizer, sep) if tokenizer is not None else None\n        self._capture = False\n        self._captured = None\n        self._hook = None\n        if self.read_layer < n:\n            self._hook = lm.model.layers[self.read_layer - 1].register_forward_hook(self._hook_fn)\n        self.last_thought_inputs = None  # (B, R*K, d) of the last think() call (for tests / analysis)\n\n    def _hook_fn(self, module, args, output):\n        if self._capture:\n            self._captured = output[0] if isinstance(output, tuple) else output\n\n    @property\n    def device(self):\n        return self.lm.get_input_embeddings().weight.device\n\n    @property\n    def emb_dtype(self):\n        return self.lm.get_input_embeddings().weight.dtype\n\n    # ------------------------------------------------------------------------------------------ low level\n    def lm_step(self, embeds, attn_mask, position_ids, cache, capture: bool):\n        """One forward over new positions. Returns (final-norm hidden (B,n,d), layer-i hidden or None)."""\n        self._capture, self._captured = capture, None\n        try:\n            out = self.lm.model(inputs_embeds=embeds, attention_mask=attn_mask, position_ids=position_ids,\n                                past_key_values=cache, use_cache=True)\n        finally:\n            self._capture = False\n        h = out.last_hidden_state\n        hi = None\n        if capture:\n            hi = h if self.read_layer == self.num_layers else self._captured\n        self._captured = None\n        return h, hi\n\n    def think(self, prompt_ids, prompt_mask, R: int):\n        """Prompt pass + R thought steps. Returns a state dict with the cache, the running 2-D mask, the next\n        position id per row and the final hidden state of the last position."""\n        dev = self.device\n        prompt_ids, prompt_mask = prompt_ids.to(dev), prompt_mask.to(dev)\n        B = prompt_ids.shape[0]\n        cache = LatentCache(self.num_layers)\n        pos = (prompt_mask.cumsum(1) - 1).clamp(min=0)\n        emb = self.lm.get_input_embeddings()(prompt_ids)\n        need_mem = self.mode == "latent" and R > 0\n        h, hi = self.lm_step(emb, prompt_mask, pos, cache, capture=need_mem)\n        mask = prompt_mask\n        next_pos = prompt_mask.sum(1)\n        last = h[:, -1]\n        thoughts = []\n        if R > 0 and self.mode in ("latent", "pause"):\n            ar = torch.arange(self.K, device=dev)\n            if self.mode == "pause":\n                soft = self.pause(B, R).to(self.emb_dtype)\n                n = soft.shape[1]\n                p = next_pos[:, None] + torch.arange(n, device=dev)\n                mask = torch.cat([mask, mask.new_ones(B, n)], 1)\n                h, _ = self.lm_step(soft, mask, p, cache, capture=False)\n                next_pos = next_pos + n\n                last = h[:, -1]\n                thoughts.append(soft)\n            else:\n                mem_h, mem_m = [hi], [prompt_mask]\n                for t in range(R):\n                    chunks = list(mem_h)\n                    if self.bptt > 0:\n                        # chunk j (j >= 1) holds thought j-1; keep grads only for the last `bptt` thoughts\n                        for j in range(1, len(chunks)):\n                            if (j - 1) < t - self.bptt:\n                                chunks[j] = chunks[j].detach()\n                    memory = torch.cat(chunks, 1)\n                    mmask = torch.cat(mem_m, 1)\n                    soft = self.compressor(memory, mmask, step=t).to(self.emb_dtype)\n                    p = next_pos[:, None] + ar\n                    mask = torch.cat([mask, mask.new_ones(B, self.K)], 1)\n                    h, hi = self.lm_step(soft, mask, p, cache, capture=True)\n                    next_pos = next_pos + self.K\n                    mem_h.append(hi)\n                    mem_m.append(mask.new_ones(B, self.K))\n                    last = h[:, -1]\n                    thoughts.append(soft)\n        self.last_thought_inputs = torch.cat(thoughts, 1) if thoughts else None\n        return dict(cache=cache, mask=mask, next_pos=next_pos, last=last)\n\n    def feed(self, state, ids, ids_mask=None):\n        """Append token ids (B, T) to the state; returns final hidden (B, T, d) and updates the state."""\n        dev = self.device\n        ids = ids.to(dev)\n        B, T = ids.shape\n        ids_mask = torch.ones_like(ids) if ids_mask is None else ids_mask.to(dev)\n        mask = torch.cat([state["mask"], ids_mask], 1)\n        p = state["next_pos"][:, None] + torch.arange(T, device=dev)\n        h, _ = self.lm_step(self.lm.get_input_embeddings()(ids), mask, p, state["cache"], capture=False)\n        state.update(mask=mask, next_pos=state["next_pos"] + T, last=h[:, -1])\n        return h\n\n    # ------------------------------------------------------------------------------------------ training\n    def build_batch(self, examples: list[dict], R: int | None = None, stage: int | None = None) -> dict:\n        R = self.R if R is None else R\n        f = self.fmt\n        plans = [plan_example(ex, self.mode, R, stage) for ex in examples]\n        r = {p[0] for p in plans}\n        assert len(r) == 1\n        prompts = [f.prompt(ex["prompt"]) for ex in examples]\n        tgts, labs = [], []\n        for ex, (_, lines) in zip(examples, plans):\n            tr = f.trace(lines)\n            ans = f.answer(ex["answer"])\n            ids = tr + f.sep + ans\n            sep_lab = f.sep if lines else [IGNORE] * len(f.sep)\n            tgts.append(ids)\n            labs.append(tr + sep_lab + ans)\n        pid, pmask = _left_pad(prompts, f.pad)\n        tid, tmask = _right_pad(tgts, f.pad)\n        lab, _ = _right_pad(labs, IGNORE, fill=IGNORE)\n        return dict(prompt_ids=pid, prompt_mask=pmask, target_ids=tid, target_mask=tmask, labels=lab,\n                    R=r.pop(), stage=stage)\n\n    def forward(self, batch: dict) -> dict:\n        """Teacher-forced loss on the target tokens. Returns dict(loss, n_tokens)."""\n        state = self.think(batch["prompt_ids"], batch["prompt_mask"], batch["R"])\n        prev_last = state["last"]  # final hidden of the last prompt/thought position: predicts target[0]\n        h = self.feed(state, batch["target_ids"], batch["target_mask"])\n        labels = batch["labels"].to(h.device)\n        pred_h = torch.cat([prev_last[:, None], h[:, :-1]], 1)\n        sel = labels != IGNORE\n        logits = self.lm.lm_head(pred_h[sel]).float()\n        return dict(loss=F.cross_entropy(logits, labels[sel]), n_tokens=int(sel.sum()))\n\n    # ------------------------------------------------------------------------------------------ inference\n    @torch.no_grad()\n    def generate(self, prompt_ids, prompt_mask, R: int, force_sep: bool, max_new_tokens: int):\n        state = self.think(prompt_ids, prompt_mask, R)\n        B = prompt_ids.shape[0]\n        if force_sep:\n            sep = torch.tensor([self.fmt.sep] * B, dtype=torch.long)\n            self.feed(state, sep)\n        out = []\n        done = torch.zeros(B, dtype=torch.bool, device=self.device)\n        for i in range(max_new_tokens):\n            nxt = self.lm.lm_head(state["last"]).argmax(-1)\n            nxt = torch.where(done, torch.full_like(nxt, self.fmt.eot), nxt)\n            out.append(nxt)\n            done |= nxt == self.fmt.eot\n            if bool(done.all()) or i == max_new_tokens - 1:\n                break\n            self.feed(state, nxt[:, None])\n        return torch.stack(out, 1).cpu()\n\n    @torch.no_grad()\n    def answer_batch(self, examples: list[dict], R: int | None = None, stage: int | None = None,\n                     max_new_tokens: int | None = None) -> list[dict]:\n        """Greedy answers. Free generation (model writes trace + \'Answer:\') when the plan keeps explicit trace\n        lines (cot / early curriculum stage); otherwise the separator is forced right after the thoughts."""\n        R = self.R if R is None else R\n        f = self.fmt\n        plans = [plan_example(ex, self.mode, R, stage) for ex in examples]\n        r_eff = plans[0][0]\n        free = any(len(p[1]) > 0 for p in plans) or self.mode == "cot"\n        if max_new_tokens is None:\n            max_new_tokens = (8 * max(len(ex["trace_steps"]) for ex in examples) + 10) if free else 4\n        pid, pmask = _left_pad([f.prompt(ex["prompt"]) for ex in examples], f.pad)\n        gen = self.generate(pid, pmask, r_eff, force_sep=not free, max_new_tokens=max_new_tokens)\n        res = []\n        for ex, g in zip(examples, gen.tolist()):\n            if f.eot in g:\n                g = g[:g.index(f.eot)]\n            text = f.tok.decode(g)\n            pred = parse_answer(text, forced_sep=not free)\n            res.append(dict(pred=pred, answer=ex["answer"], correct=pred == ex["answer"], text=text))\n        return res\n\n    @torch.no_grad()\n    def greedy_answer(self, prompt: str | list[str], R: int | None = None, max_new_tokens: int = 4):\n        """Answer raw program prompt(s) with R thought steps (R may differ from training R) and a forced\n        separator. Returns int | None (or a list for a list input)."""\n        single = isinstance(prompt, str)\n        prompts = [prompt] if single else list(prompt)\n        exs = [dict(prompt=p, answer=None, trace_steps=[]) for p in prompts]\n        mode, self.mode = self.mode, ("none" if self.mode == "cot" else self.mode)\n        try:\n            out = [r["pred"] for r in self.answer_batch(exs, R=0 if self.mode == "none" else R,\n                                                         max_new_tokens=max_new_tokens)]\n        finally:\n            self.mode = mode\n        return out[0] if single else out\n\n\ndef default_read_layer(num_layers: int) -> int:\n    return max(1, min(num_layers, round(2 * num_layers / 3)))\n\n',
    'bitnet_vl/latent/train_latent.py': '"""Train / evaluate the latent-reasoning experiment.\n\n    cd multimodal\n    # tiny random BitNet on CPU (sanity check; trains all weights)\n    python -m bitnet_vl.latent.train_latent --model tiny --train-base --mode latent --R 2 --K 2 \\\\\n        --hops-train 2-3 --hops-test 2-3 --steps 1500 --out runs/tiny_latent\n    # real model (GPU), LoRA + compressor\n    python -m bitnet_vl.latent.train_latent --model microsoft/bitnet-b1.58-2B-4T-bf16 --dtype bf16 \\\\\n        --mode latent --curriculum --steps-per-stage 300 --R 6 --K 4 --hops-train 2-6 --hops-test 2-8 \\\\\n        --steps 2400 --batch-size 8 --grad-accum 2 --grad-checkpointing --out runs/latent\n\nImportable: ``run(config: dict) -> results dict`` (keys as the CLI flags with dashes -> underscores). Writes\n``<out>/results.json`` (accuracy per (eval R, hops), loss curve, periodic evals, config, throughput).\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom contextlib import nullcontext\nfrom pathlib import Path\n\nimport torch\n\nfrom ..bitlinear import replace_linears\nfrom ..modeling import LLM_LINEAR_RE, _strip_quant_config\nfrom .compressor import LatentCompressor, PauseTokens, embedding_rms\nfrom .lora import apply_lora, count_params, freeze_base, lora_parameters, save_trainable\nfrom .reasoner import LatentReasoner, default_read_layer, enable_checkpointing, num_stages\nfrom .tasks import TaskSampler, make_test_set, parse_hops\n\nDTYPES = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}\n\nDEFAULTS = dict(\n    model="tiny", mode="latent", R=4, K=4, read_layer=None, hops_train="2-6", hops_test="2-8", distractors=2,\n    steps=1000, batch_size=8, grad_accum=1, lr_compressor=1e-3, lr_lora=2e-4, lr_base=1e-3, weight_decay=0.0,\n    warmup_steps=50, lora_rank=16, lora_alpha=32.0, lora_dropout=0.0, train_base=False, curriculum=False,\n    steps_per_stage=200, bptt=0, dtype="fp32", grad_checkpointing=False, eval_every=0, eval_quick_n=32,\n    eval_R_sweep="0,1,2,4,6,8", n_test=100, eval_batch_size=32, compressor_layers=1, compressor_heads=8,\n    max_grad_norm=1.0, seed=0, out=None, device=None, log_every=25, save_weights=False,\n    tiny_hidden=256, tiny_layers=4, tiny_intermediate=1024, tiny_heads=4, tiny_kv_heads=2, threads=0,\n    max_minutes=0.0,\n)\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# model construction\n# ----------------------------------------------------------------------------------------------------------------\ndef _fp16_safe_mlp(mlp):\n    """relu^2 * up can overflow fp16; compute the product in fp32 (ffn_sub_norm then brings it back to O(1))."""\n\n    def forward(x):\n        g = mlp.gate_proj(x).float()\n        u = mlp.up_proj(x).float()\n        return mlp.down_proj(mlp.ffn_sub_norm(mlp.act_fn(g) * u))\n\n    mlp.forward = forward\n\n\n_TOK_CACHE = {}\n\n\ndef _tiny_tokenizer():\n    """Llama-3 tokenizer for the tiny model: rebuilt offline from llama.cpp\'s vocab GGUF when it is available (repo\n    checkout; override the path with $BITNET_VOCAB_GGUF), else downloaded from the hub ($BITNET_TOKENIZER, default\n    the 2B4T repo; same 128256-token vocabulary). Cached: building it from the GGUF takes a few seconds."""\n    if "tok" not in _TOK_CACHE:\n        from ..tokenizer_utils import DEFAULT_VOCAB_GGUF, build_llama3_tokenizer\n\n        gguf = Path(os.environ.get("BITNET_VOCAB_GGUF", DEFAULT_VOCAB_GGUF))\n        if gguf.exists():\n            _TOK_CACHE["tok"] = build_llama3_tokenizer(gguf)\n        else:\n            from transformers import AutoTokenizer\n\n            _TOK_CACHE["tok"] = AutoTokenizer.from_pretrained(\n                os.environ.get("BITNET_TOKENIZER", "microsoft/bitnet-b1.58-2B-4T-bf16"))\n    return _TOK_CACHE["tok"]\n\n\ndef load_lm(cfg: dict, device):\n    """-> (lm with BitLinear decoder linears, tokenizer). Master weights fp32 when the base is trained, else the\n    requested dtype (fp16/bf16 halves the memory of the frozen base)."""\n    from transformers import AutoConfig, AutoTokenizer, BitNetForCausalLM\n\n    train_base = cfg["train_base"]\n    store_dtype = torch.float32 if (train_base or device.type == "cpu" and cfg["dtype"] == "fp16") \\\n        else DTYPES[cfg["dtype"]]\n    if cfg["model"] == "tiny":\n        from ..make_tiny_ckpt import tiny_llm_config\n        tok = _tiny_tokenizer()\n        torch.manual_seed(cfg["seed"])\n        lcfg = tiny_llm_config(vocab_size=len(tok), hidden=cfg["tiny_hidden"], intermediate=cfg["tiny_intermediate"],\n                               layers=cfg["tiny_layers"], heads=cfg["tiny_heads"], kv_heads=cfg["tiny_kv_heads"])\n        lm = BitNetForCausalLM(_strip_quant_config(lcfg)).to(store_dtype)\n    else:\n        lcfg = _strip_quant_config(AutoConfig.from_pretrained(cfg["model"]))\n        lm = BitNetForCausalLM.from_pretrained(cfg["model"], config=lcfg, dtype=store_dtype)\n        tok = AutoTokenizer.from_pretrained(cfg["model"])\n    lm.config.use_cache = True\n    replace_linears(lm, lambda n, m: bool(LLM_LINEAR_RE.search(n)))  # BitLinear QAT forward (W1.58 A8)\n    lm.to(device)\n    if cfg["lora_rank"] > 0:\n        apply_lora(lm, r=cfg["lora_rank"], alpha=cfg["lora_alpha"], dropout=cfg["lora_dropout"],\n                   prequantize=not train_base)\n    if train_base:\n        lm.requires_grad_(True)\n    else:\n        freeze_base(lm)\n    if cfg["dtype"] == "fp16" and device.type == "cuda":\n        for layer in lm.model.layers:\n            _fp16_safe_mlp(layer.mlp)\n    if cfg["grad_checkpointing"]:\n        enable_checkpointing(lm, True)\n    return lm, tok\n\n\ndef build_reasoner(cfg: dict, lm, tok, device) -> LatentReasoner:\n    d = lm.config.hidden_size\n    rms = embedding_rms(lm.get_input_embeddings().weight)\n    comp = pause = None\n    if cfg["mode"] == "latent":\n        heads = cfg["compressor_heads"]\n        while d % heads:\n            heads //= 2\n        comp = LatentCompressor(d, cfg["K"], n_layers=cfg["compressor_layers"], n_heads=heads, target_rms=rms,\n                                max_steps=max(16, cfg["R"] + 1)).to(device)\n    elif cfg["mode"] == "pause":\n        g = torch.Generator().manual_seed(cfg["seed"])\n        idx = torch.randint(0, lm.get_input_embeddings().num_embeddings, (cfg["K"],), generator=g)\n        pause = PauseTokens(d, cfg["K"], target_rms=rms,\n                            init=lm.get_input_embeddings().weight[idx.to(device)].float()).to(device)\n    rl = cfg["read_layer"] or default_read_layer(lm.config.num_hidden_layers)\n    return LatentReasoner(lm, comp, read_layer=rl, K=cfg["K"], mode=cfg["mode"], tokenizer=tok, pause=pause,\n                          R=cfg["R"], bptt=cfg["bptt"])\n\n\ndef param_groups(cfg, reasoner):\n    groups = []\n    aux = [p for m in (reasoner.compressor, reasoner.pause) if m is not None for p in m.parameters()]\n    if aux:\n        groups.append(dict(params=aux, lr=cfg["lr_compressor"], name="compressor", weight_decay=cfg["weight_decay"]))\n    lora = [p for p in lora_parameters(reasoner.lm) if p.requires_grad]\n    lora_ids = {id(p) for p in lora}\n    if lora:\n        groups.append(dict(params=lora, lr=cfg["lr_lora"], name="lora", weight_decay=0.0))\n    base = [p for p in reasoner.lm.parameters() if p.requires_grad and id(p) not in lora_ids]\n    if base:\n        groups.append(dict(params=base, lr=cfg["lr_base"], name="base", weight_decay=cfg["weight_decay"]))\n    for g in groups:\n        g["base_lr"] = g["lr"]\n    return groups\n\n\ndef lr_factor(step, total, warmup):\n    if step < warmup:\n        return (step + 1) / max(1, warmup)\n    p = (step - warmup) / max(1, total - warmup)\n    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# evaluation\n# ----------------------------------------------------------------------------------------------------------------\ndef evaluate(reasoner, test_set, R, stage=None, batch_size=32, limit=None, autocast=nullcontext):\n    was = reasoner.training\n    reasoner.eval()\n    acc = {}\n    samples = {}\n    with autocast():\n        for h, exs in test_set.items():\n            exs = exs[:limit] if limit else exs\n            res = []\n            for i in range(0, len(exs), batch_size):\n                res += reasoner.answer_batch(exs[i:i + batch_size], R=R, stage=stage)\n            acc[str(h)] = sum(r["correct"] for r in res) / max(1, len(res))\n            samples[str(h)] = res[0]["text"]\n    reasoner.train(was)\n    return acc, samples\n\n\ndef eval_R_values(cfg) -> list[int]:\n    if cfg["mode"] in ("none", "cot"):\n        return [0]\n    vals = sorted({int(x) for x in str(cfg["eval_R_sweep"]).split(",") if x != ""} | {cfg["R"]})\n    return vals\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# training\n# ----------------------------------------------------------------------------------------------------------------\ndef run(config: dict) -> dict:\n    cfg = dict(DEFAULTS)\n    unknown = set(config) - set(DEFAULTS)\n    if unknown:\n        raise ValueError(f"unknown config keys: {sorted(unknown)}")\n    cfg.update(config)\n    if cfg["threads"]:\n        torch.set_num_threads(cfg["threads"])\n    device = torch.device(cfg["device"] or ("cuda" if torch.cuda.is_available() else "cpu"))\n    random.seed(cfg["seed"])\n    torch.manual_seed(cfg["seed"])\n    t0 = time.time()\n    lm, tok = load_lm(cfg, device)\n    reasoner = build_reasoner(cfg, lm, tok, device)\n    cfg["read_layer"] = reasoner.read_layer\n    n_trainable = count_params(reasoner)\n    print(f"[latent] model={cfg[\'model\']} mode={cfg[\'mode\']} R={cfg[\'R\']} K={cfg[\'K\']} read_layer="\n          f"{reasoner.read_layer}/{reasoner.num_layers} trainable={n_trainable / 1e6:.2f}M device={device} "\n          f"load {time.time() - t0:.1f}s", flush=True)\n\n    amp_dtype = DTYPES[cfg["dtype"]]\n    use_amp = amp_dtype != torch.float32\n    autocast = (lambda: torch.autocast(device.type, dtype=amp_dtype)) if use_amp else nullcontext\n    scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and amp_dtype == torch.float16 and device.type == "cuda"))\n    groups = param_groups(cfg, reasoner)\n    if not groups:\n        raise ValueError("nothing to train: use --lora-rank > 0 or --train-base")\n    opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), eps=1e-8) if groups else None\n    trainable = [p for g in groups for p in g["params"]]\n\n    hops_train, hops_test = parse_hops(cfg["hops_train"]), parse_hops(cfg["hops_test"])\n    sampler = TaskSampler(hops_train, cfg["distractors"], seed=cfg["seed"])\n    test_set = make_test_set(hops_test, cfg["n_test"], cfg["distractors"], seed=cfg["seed"])\n    curriculum = cfg["curriculum"] and cfg["mode"] in ("latent", "pause")\n    n_st = num_stages(cfg["R"]) if curriculum else 1\n\n    def stage_at(step):\n        return min(step // cfg["steps_per_stage"], n_st - 1) if curriculum else None\n\n    loss_curve, evals = [], []\n    reasoner.train()\n    t_train = time.time()\n    n_examples = n_positions = 0\n    run_loss, run_n = 0.0, 0\n    step = 0\n    stopped_early = False\n    for step in range(cfg["steps"]):\n        stage = stage_at(step)\n        f = lr_factor(step, cfg["steps"], cfg["warmup_steps"])\n        for g in opt.param_groups:\n            g["lr"] = g["base_lr"] * f\n        tot = 0.0\n        for _ in range(cfg["grad_accum"]):\n            batch = reasoner.build_batch(sampler.batch(cfg["batch_size"]), stage=stage)\n            with autocast():\n                out = reasoner(batch)\n            loss = out["loss"] / cfg["grad_accum"]\n            scaler.scale(loss).backward()\n            tot += loss.item()\n            n_examples += cfg["batch_size"]\n            n_positions += cfg["batch_size"] * (batch["prompt_ids"].shape[1] + batch["R"] * cfg["K"] *\n                                                (cfg["mode"] in ("latent", "pause")) + batch["target_ids"].shape[1])\n        scaler.unscale_(opt)\n        gn = torch.nn.utils.clip_grad_norm_(trainable, cfg["max_grad_norm"])\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n        run_loss += tot\n        run_n += 1\n        if (step + 1) % cfg["log_every"] == 0 or step == 0:\n            avg = run_loss / run_n\n            loss_curve.append(dict(step=step + 1, loss=avg, stage=stage, grad_norm=float(gn)))\n            el = time.time() - t_train\n            print(f"[latent] step {step + 1}/{cfg[\'steps\']} stage {stage} loss {avg:.4f} gn {float(gn):.2f} "\n                  f"{n_examples / el:.1f} ex/s", flush=True)\n            run_loss, run_n = 0.0, 0\n        if cfg["eval_every"] and (step + 1) % cfg["eval_every"] == 0 and step + 1 < cfg["steps"]:\n            acc, smp = evaluate(reasoner, test_set, cfg["R"], stage, cfg["eval_batch_size"], cfg["eval_quick_n"],\n                                autocast)\n            evals.append(dict(step=step + 1, stage=stage, R=cfg["R"], acc=acc))\n            print(f"[latent] eval step {step + 1} stage {stage}: {acc}  e.g. {smp[next(iter(smp))]!r}", flush=True)\n        if cfg["max_minutes"] and (time.time() - t_train) > 60 * cfg["max_minutes"]:\n            print(f"[latent] time budget reached at step {step + 1}", flush=True)\n            stopped_early = True\n            break\n    train_time = time.time() - t_train\n    steps_done = step + 1\n\n    # final evaluation: all-latent setting, eval-R sweep\n    t_eval = time.time()\n    final, final_samples = {}, {}\n    for R in eval_R_values(cfg):\n        acc, smp = evaluate(reasoner, test_set, R, None, cfg["eval_batch_size"], None, autocast)\n        final[str(R)] = acc\n        final_samples[str(R)] = smp\n        print(f"[latent] final eval R={R}: {acc}", flush=True)\n    eval_time = time.time() - t_eval\n    results = dict(\n        config=cfg, final_acc=final, final_samples=final_samples, loss_curve=loss_curve, evals=evals,\n        throughput=dict(train_seconds=train_time, steps=steps_done, examples_per_s=n_examples / max(train_time, 1e-9),\n                        positions_per_s=n_positions / max(train_time, 1e-9), eval_seconds=eval_time,\n                        stopped_early=stopped_early),\n        n_trainable=n_trainable, device=str(device),\n        max_memory_gb=(torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else None),\n    )\n    if cfg["out"]:\n        out = Path(cfg["out"])\n        out.mkdir(parents=True, exist_ok=True)\n        with open(out / "results.json", "w") as fh:\n            json.dump(results, fh, indent=1)\n        if cfg["save_weights"]:\n            save_trainable(out / "trainable.pt", lm, {"compressor": reasoner.compressor, "pause": reasoner.pause},\n                           include_base=cfg["train_base"])\n        print(f"[latent] wrote {out / \'results.json\'}", flush=True)\n    results["_reasoner"] = reasoner  # not serialized; handy for notebooks (pop it before json.dump)\n    return results\n\n\ndef parse_args(argv=None) -> dict:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    D = DEFAULTS\n    ap.add_argument("--model", default=D["model"], help="\'tiny\' (random tiny BitNet) or an HF id / local dir")\n    ap.add_argument("--mode", choices=["none", "latent", "pause", "cot"], default=D["mode"])\n    ap.add_argument("--R", type=int, default=D["R"], help="thought steps (final curriculum stage)")\n    ap.add_argument("--K", type=int, default=D["K"], help="soft tokens per thought step")\n    ap.add_argument("--read-layer", type=int, default=None, help="HF hidden_states index (default round(2L/3))")\n    ap.add_argument("--hops-train", default=D["hops_train"])\n    ap.add_argument("--hops-test", default=D["hops_test"])\n    ap.add_argument("--distractors", type=int, default=D["distractors"])\n    ap.add_argument("--steps", type=int, default=D["steps"], help="optimizer steps")\n    ap.add_argument("--batch-size", type=int, default=D["batch_size"])\n    ap.add_argument("--grad-accum", type=int, default=D["grad_accum"])\n    ap.add_argument("--lr", type=float, default=None, help="shortcut: sets --lr-lora and --lr-base")\n    ap.add_argument("--lr-compressor", type=float, default=D["lr_compressor"], help="compressor / pause embeddings")\n    ap.add_argument("--lr-lora", type=float, default=D["lr_lora"])\n    ap.add_argument("--lr-base", type=float, default=D["lr_base"], help="with --train-base")\n    ap.add_argument("--weight-decay", type=float, default=D["weight_decay"])\n    ap.add_argument("--warmup-steps", type=int, default=D["warmup_steps"])\n    ap.add_argument("--lora-rank", type=int, default=D["lora_rank"], help="0 disables LoRA")\n    ap.add_argument("--lora-alpha", type=float, default=D["lora_alpha"])\n    ap.add_argument("--lora-dropout", type=float, default=D["lora_dropout"])\n    ap.add_argument("--train-base", action="store_true", help="train ALL LLM weights (needed for --model tiny)")\n    ap.add_argument("--curriculum", action="store_true", help="Coconut curriculum (latent / pause modes)")\n    ap.add_argument("--steps-per-stage", type=int, default=D["steps_per_stage"])\n    ap.add_argument("--bptt", type=int, default=D["bptt"], help="keep grads through the last N thoughts (0 = all)")\n    ap.add_argument("--dtype", choices=list(DTYPES), default=D["dtype"])\n    ap.add_argument("--grad-checkpointing", action="store_true")\n    ap.add_argument("--eval-every", type=int, default=D["eval_every"])\n    ap.add_argument("--eval-quick-n", type=int, default=D["eval_quick_n"], help="examples per hops in periodic evals")\n    ap.add_argument("--eval-R-sweep", default=D["eval_R_sweep"])\n    ap.add_argument("--n-test", type=int, default=D["n_test"], help="test examples per hops")\n    ap.add_argument("--eval-batch-size", type=int, default=D["eval_batch_size"])\n    ap.add_argument("--compressor-layers", type=int, default=D["compressor_layers"])\n    ap.add_argument("--compressor-heads", type=int, default=D["compressor_heads"])\n    ap.add_argument("--max-grad-norm", type=float, default=D["max_grad_norm"])\n    ap.add_argument("--seed", type=int, default=D["seed"])\n    ap.add_argument("--device", default=None)\n    ap.add_argument("--log-every", type=int, default=D["log_every"])\n    ap.add_argument("--save-weights", action="store_true", help="save LoRA + compressor (or full LM with --train-base)")\n    ap.add_argument("--threads", type=int, default=0)\n    ap.add_argument("--max-minutes", type=float, default=0.0, help="stop training after this wall time (0 = off)")\n    ap.add_argument("--tiny-hidden", type=int, default=D["tiny_hidden"])\n    ap.add_argument("--tiny-layers", type=int, default=D["tiny_layers"])\n    ap.add_argument("--tiny-intermediate", type=int, default=D["tiny_intermediate"])\n    ap.add_argument("--out", default=None)\n    a = vars(ap.parse_args(argv))\n    lr = a.pop("lr")\n    if lr is not None:\n        a["lr_lora"] = a["lr_base"] = lr\n    return a\n\n\ndef main(argv=None):\n    res = run(parse_args(argv))\n    res.pop("_reasoner", None)\n\n\nif __name__ == "__main__":\n    main()\n',
    'bitnet_vl/tests/test_latent.py': '"""CPU tests for the latent-reasoning experiment.  Run:  cd multimodal && python -m pytest bitnet_vl/tests/test_latent.py -q"""\n\nimport copy\nimport random\n\nimport pytest\nimport torch\nimport torch.nn as nn\n\nfrom bitnet_vl.bitlinear import BitLinear, replace_linears\nfrom bitnet_vl.latent import train_latent\nfrom bitnet_vl.latent.compressor import LatentCompressor, PauseTokens, embedding_rms\nfrom bitnet_vl.latent.lora import LoRALinear, apply_lora, count_params, freeze_base, load_trainable, save_trainable\nfrom bitnet_vl.latent.reasoner import IGNORE, LatentReasoner, enable_checkpointing, plan_example\nfrom bitnet_vl.latent.tasks import (TaskSampler, exec_program, make_example, make_test_set, parse_hops,\n                                    trace_values)\nfrom bitnet_vl.make_tiny_ckpt import tiny_llm_config\nfrom bitnet_vl.modeling import LLM_LINEAR_RE, _strip_quant_config\n\ntorch.set_num_threads(4)\nD, K = 128, 2\n\n\n@pytest.fixture(scope="session")\ndef tok():\n    return train_latent._tiny_tokenizer()\n\n\ndef _tiny_lm(tok, seed=0, layers=3):\n    from transformers import BitNetForCausalLM\n\n    torch.manual_seed(seed)\n    cfg = tiny_llm_config(vocab_size=len(tok), hidden=D, intermediate=256, layers=layers, heads=4, kv_heads=2)\n    lm = BitNetForCausalLM(_strip_quant_config(cfg))\n    replace_linears(lm, lambda n, m: bool(LLM_LINEAR_RE.search(n)))\n    with torch.no_grad():  # non-trivial norms\n        for n, p in lm.named_parameters():\n            if p.ndim == 1:\n                p.add_(0.1 * torch.randn_like(p))\n    return lm.eval()\n\n\ndef _reasoner(tok, mode="latent", R=3, lora=True, seed=0, prequantize=True, bptt=0):\n    lm = _tiny_lm(tok, seed)\n    if lora:\n        apply_lora(lm, r=4, alpha=8, prequantize=prequantize)\n        freeze_base(lm)\n        with torch.no_grad():  # LoRA B is zero-init; make the LoRA path active for the tests\n            for n, p in lm.named_parameters():\n                if n.endswith("lora_B"):\n                    p.normal_(0, 0.02)\n    rms = embedding_rms(lm.get_input_embeddings().weight)\n    comp = LatentCompressor(D, K, n_layers=1, n_heads=4, target_rms=rms) if mode == "latent" else None\n    pause = PauseTokens(D, K, target_rms=rms) if mode == "pause" else None\n    return LatentReasoner(lm, comp, read_layer=2, K=K, mode=mode, tokenizer=tok, pause=pause, R=R, bptt=bptt).eval()\n\n\ndef _set_act_quant(r, on: bool):\n    """Per-token int8 rounding is discontinuous: a 1e-7 difference between two mathematically equal computations\n    (different matmul shapes) can flip one rounding and move logits by ~1e-3. Exact-equivalence tests therefore run\n    with activation quantization off, plus a loose check with it on."""\n    for m in r.modules():\n        if isinstance(m, BitLinear):\n            m.quantize_activations = on\n\n\ndef _examples(n=3, seed=0, hops=(2, 3, 4), distractors=2):\n    rng = random.Random(seed)\n    return [make_example(rng, hops[i % len(hops)], distractors) for i in range(n)]\n\n\n# ------------------------------------------------------------------------------------------------------- tasks\ndef test_tasks_match_exec_and_deterministic():\n    rng = random.Random(123)\n    for _ in range(500):\n        h, d = rng.randint(1, 8), rng.randint(0, 6)\n        ex = make_example(rng, h, d)\n        assert exec_program(ex["prompt"]) == ex["answer"]\n        assert 0 <= ex["answer"] <= 99\n        assert len(ex["trace_steps"]) == h == ex["hops"]\n        assert trace_values(ex["prompt"], ex["trace_steps"])\n        assert ex["trace_steps"][-1] == f"{ex[\'query\']} = {ex[\'answer\']}"\n        assert ex["prompt"].strip().endswith(f"print({ex[\'query\']})")\n        assert len(ex["program_lines"]) == h + 1 + d + 1\n    a = TaskSampler([2, 3], 2, seed=5).batch(10)\n    b = TaskSampler([2, 3], 2, seed=5).batch(10)\n    assert a == b\n    assert TaskSampler([2, 3], 2, seed=6).batch(10) != a\n    test = make_test_set([2, 8], 20, 2)\n    assert set(test) == {2, 8} and all(e["hops"] == 8 for e in test[8])\n    assert test == make_test_set([2, 8], 20, 2)\n    train_prompts = {e["prompt"] for e in TaskSampler([2], 2, seed=0).batch(200)}\n    assert len(train_prompts & {e["prompt"] for e in test[2]}) < 3  # disjoint seed spaces\n    assert parse_hops("2-4,7") == [2, 3, 4, 7]\n\n\n# ------------------------------------------------------------------------------------------------------- compressor\ndef test_compressor_output_rms_matches_embeddings(tok):\n    lm = _tiny_lm(tok)\n    rms = embedding_rms(lm.get_input_embeddings().weight)\n    comp = LatentCompressor(D, 3, n_layers=2, n_heads=4, target_rms=rms)\n    mem = torch.randn(2, 11, D) * 50  # hidden states with a very different scale\n    mask = torch.ones(2, 11, dtype=torch.long)\n    mask[0, :4] = 0\n    out = comp(mem, mask, step=1)\n    assert out.shape == (2, 3, D)\n    tok_rms = out.pow(2).mean(-1).sqrt()\n    torch.testing.assert_close(tok_rms, torch.full_like(tok_rms, rms), rtol=1e-4, atol=1e-5)\n    # masked memory entries are ignored\n    mem2 = mem.clone()\n    mem2[0, :4] = 1e3 * torch.randn(4, D)\n    torch.testing.assert_close(comp(mem2, mask, step=1)[0], out[0])\n    p = PauseTokens(D, 3, target_rms=rms)(2, 4)\n    assert p.shape == (2, 12, D)\n    torch.testing.assert_close(p.pow(2).mean(-1).sqrt(), torch.full((2, 12), rms), rtol=1e-4, atol=1e-5)\n\n\n# ------------------------------------------------------------------------------------------------------- KV cache\ndef _full_forward(r, prompt_ids, prompt_mask, thoughts, target_ids, target_mask):\n    """No-cache reference: one forward over [prompt embeds ; thought embeds ; target embeds]."""\n    emb = r.lm.get_input_embeddings()\n    parts = [emb(prompt_ids)] + ([thoughts] if thoughts is not None else []) + [emb(target_ids)]\n    x = torch.cat(parts, 1)\n    n_th = 0 if thoughts is None else thoughts.shape[1]\n    B = prompt_ids.shape[0]\n    mask = torch.cat([prompt_mask, prompt_mask.new_ones(B, n_th), target_mask], 1)\n    pos = (mask.cumsum(1) - 1).clamp(min=0)\n    out = r.lm(inputs_embeds=x, attention_mask=mask, position_ids=pos, use_cache=False, output_hidden_states=True)\n    return out, prompt_ids.shape[1], n_th\n\n\n@pytest.mark.parametrize("act_quant", [False, True])\n@pytest.mark.parametrize("mode", ["latent", "pause"])\ndef test_kv_cache_loop_matches_full_forward(tok, mode, act_quant):\n    torch.manual_seed(0)\n    r = _reasoner(tok, mode, R=3)\n    _set_act_quant(r, act_quant)\n    tol = dict(atol=1e-4, rtol=1e-4) if not act_quant else dict(atol=5e-2, rtol=0)\n    batch = r.build_batch(_examples(3))\n    with torch.no_grad():\n        state = r.think(batch["prompt_ids"], batch["prompt_mask"], 3)\n        prev = state["last"]\n        h = r.feed(state, batch["target_ids"], batch["target_mask"])\n        logits_cached = r.lm.lm_head(torch.cat([prev[:, None], h[:, :-1]], 1))\n        thoughts = r.last_thought_inputs\n        assert thoughts.shape[1] == 3 * K\n        out, P, n_th = _full_forward(r, batch["prompt_ids"], batch["prompt_mask"], thoughts, batch["target_ids"],\n                                     batch["target_mask"])\n        ref = out.logits[:, P + n_th - 1: P + n_th - 1 + batch["target_ids"].shape[1]]\n        valid = batch["target_mask"].bool()\n        torch.testing.assert_close(logits_cached[valid], ref[valid], **tol)\n        if mode == "latent":\n            # the soft tokens are what the compressor makes of the (no-cache) layer-i states of everything before\n            hs = out.hidden_states[r.read_layer]\n            mmask = torch.cat([batch["prompt_mask"], batch["prompt_mask"].new_ones(3, n_th)], 1)\n            for t in range(3):\n                L = P + t * K\n                soft = r.compressor(hs[:, :L], mmask[:, :L], step=t)\n                torch.testing.assert_close(soft, thoughts[:, t * K:(t + 1) * K], **tol)\n\n\ndef test_hidden_capture_matches_output_hidden_states(tok):\n    r = _reasoner(tok, "latent", R=1)\n    b = r.build_batch(_examples(2))\n    with torch.no_grad():\n        emb = r.lm.get_input_embeddings()(b["prompt_ids"])\n        pos = (b["prompt_mask"].cumsum(1) - 1).clamp(min=0)\n        from bitnet_vl.latent.reasoner import LatentCache\n\n        _, hi = r.lm_step(emb, b["prompt_mask"], pos, LatentCache(r.num_layers), capture=True)\n        ref = r.lm(inputs_embeds=emb, attention_mask=b["prompt_mask"], position_ids=pos,\n                   output_hidden_states=True).hidden_states[r.read_layer]\n    torch.testing.assert_close(hi, ref)\n\n\n@pytest.mark.parametrize("act_quant", [False, True])\n@pytest.mark.parametrize("mode", ["latent", "pause", "none"])\ndef test_left_padded_batch_matches_unbatched(tok, mode, act_quant):\n    torch.manual_seed(0)\n    r = _reasoner(tok, mode, R=2)\n    _set_act_quant(r, act_quant)\n    tol = dict(atol=1e-4, rtol=1e-4) if not act_quant else dict(atol=5e-2, rtol=0)\n    exs = _examples(3, hops=(2, 5, 3), distractors=3)\n\n    def answer_logits(examples):\n        b = r.build_batch(examples)\n        with torch.no_grad():\n            st = r.think(b["prompt_ids"], b["prompt_mask"], b["R"])\n            prev = st["last"]\n            h = r.feed(st, b["target_ids"], b["target_mask"])\n            lg = r.lm.lm_head(torch.cat([prev[:, None], h[:, :-1]], 1))\n        return [lg[i, b["labels"][i] != IGNORE] for i in range(len(examples))]\n\n    batched = answer_logits(exs)\n    assert len({len(r.fmt.prompt(e["prompt"])) for e in exs}) > 1  # padding actually happens\n    for i, e in enumerate(exs):\n        single = answer_logits([e])[0]\n        torch.testing.assert_close(batched[i], single, **tol)\n    if not act_quant:  # greedy decoding agrees too (with act quant a near-tie argmax of a random model could flip)\n        assert [x["text"] for x in r.answer_batch(exs)] == [r.answer_batch([e])[0]["text"] for e in exs]\n\n\n# ------------------------------------------------------------------------------------------------------- grads\ndef test_gradients_reach_compressor_and_lora_base_frozen(tok):\n    r = _reasoner(tok, "latent", R=2).train()\n    out = r(r.build_batch(_examples(2)))\n    out["loss"].backward()\n    for n, p in r.compressor.named_parameters():\n        assert p.grad is not None and p.grad.abs().sum() > 0, n\n    lora = [(n, p) for n, p in r.lm.named_parameters() if "lora_" in n]\n    assert lora and all(p.grad is not None and p.grad.abs().sum() > 0 for n, p in lora)\n    base = [(n, p) for n, p in r.lm.named_parameters() if "lora_" not in n]\n    assert all(not p.requires_grad and p.grad is None for n, p in base)\n    assert count_params(r.lm) == sum(p.numel() for _, p in lora)\n\n\ndef test_checkpointing_gives_same_grads(tok):\n    grads = []\n    for ckpt in (False, True):\n        r = _reasoner(tok, "latent", R=2, seed=1).train()\n        if ckpt:\n            enable_checkpointing(r.lm, True)\n        torch.manual_seed(0)\n        out = r(r.build_batch(_examples(2, seed=3)))\n        out["loss"].backward()\n        grads.append((out["loss"].item(), {n: p.grad.clone() for n, p in r.named_parameters() if p.grad is not None}))\n    assert abs(grads[0][0] - grads[1][0]) < 1e-5\n    assert grads[0][1].keys() == grads[1][1].keys()\n    for n in grads[0][1]:\n        torch.testing.assert_close(grads[0][1][n], grads[1][1][n], atol=1e-5, rtol=1e-4, msg=n)\n\n\ndef test_bptt_truncation_cuts_old_thought_paths(tok):\n    # with bptt=1, thought 0\'s memory entry is detached when making thought 2: grads differ from full BPTT but exist\n    g = []\n    for bptt in (0, 1):\n        r = _reasoner(tok, "latent", R=3, seed=2, bptt=bptt).train()\n        r(r.build_batch(_examples(2, seed=4)))["loss"].backward()\n        g.append(r.compressor.queries.grad.clone())\n    assert g[1].abs().sum() > 0 and not torch.allclose(g[0], g[1])\n\n\ndef test_pause_has_no_dependence_on_hidden_states(tok):\n    r = _reasoner(tok, "pause", R=2)\n    with torch.no_grad():\n        b1 = r.build_batch(_examples(2, seed=0))\n        r.think(b1["prompt_ids"], b1["prompt_mask"], 2)\n        t1 = r.last_thought_inputs.clone()\n        b2 = r.build_batch(_examples(2, seed=9))\n        r.think(b2["prompt_ids"], b2["prompt_mask"], 2)\n        t2 = r.last_thought_inputs.clone()\n    torch.testing.assert_close(t1, t2)\n    assert t1.shape == (2, 2 * K, D)\n    torch.testing.assert_close(t1[:, :K], t1[:, K:])  # same pause block every step\n    r.train()\n    r(r.build_batch(_examples(2)))["loss"].backward()\n    assert r.pause.emb.grad is not None and r.pause.emb.grad.abs().sum() > 0\n\n\n# ------------------------------------------------------------------------------------------------------- curriculum\ndef test_curriculum_stage_construction(tok):\n    ex = dict(prompt="a = 1\\nb = a + 1\\nc = b * 2\\nprint(c)\\n", answer=4, trace_steps=["b = 2", "c = 4"])\n    assert plan_example(ex, "none", 3) == (0, [])\n    assert plan_example(ex, "cot", 3) == (0, ["b = 2", "c = 4"])\n    assert plan_example(ex, "latent", 3) == (3, [])\n    assert plan_example(ex, "latent", 3, stage=0) == (0, ["b = 2", "c = 4"])\n    assert plan_example(ex, "latent", 3, stage=1) == (1, ["c = 4"])\n    assert plan_example(ex, "latent", 3, stage=2) == (2, [])\n    assert plan_example(ex, "latent", 3, stage=3) == (3, [])\n    assert plan_example(ex, "pause", 3, stage=7) == (3, [])\n    r = _reasoner(tok, "latent", R=3)\n    f = r.fmt\n    b = r.build_batch([ex], stage=1)\n    assert b["R"] == 1\n    tgt = b["target_ids"][0].tolist()\n    lab = b["labels"][0].tolist()\n    exp = f.trace(["c = 4"]) + f.sep + f.answer(4)\n    assert tgt == exp and lab == exp  # remaining trace + separator + answer all in the loss\n    assert f.tok.decode(tgt) == "c = 4\\nAnswer: 4<|eot_id|>"\n    b = r.build_batch([ex], stage=3)\n    assert b["R"] == 3\n    lab = b["labels"][0].tolist()\n    assert lab[:len(f.sep)] == [IGNORE] * len(f.sep) and lab[len(f.sep):] == f.answer(4)\n    assert f.tok.decode(b["prompt_ids"][0]) == "<|begin_of_text|>" + ex["prompt"]\n\n\n# ------------------------------------------------------------------------------------------------------- lora\ndef test_lora_prequantize_matches_bitlinear_and_roundtrip(tmp_path):\n    torch.manual_seed(0)\n    bl = BitLinear(128, 64, bias=False)\n    x = torch.randn(3, 5, 128)\n    ref = bl(x)\n    a, b = LoRALinear(copy.deepcopy(bl), r=4), LoRALinear(copy.deepcopy(bl), r=4)\n    b.prequantize_base_()\n    torch.testing.assert_close(a(x), ref)\n    torch.testing.assert_close(b(x), ref, atol=1e-5, rtol=1e-5)\n    m = nn.Sequential()\n    m.add_module("q_proj", LoRALinear(nn.Linear(8, 8), r=2))\n    nn.init.normal_(m.q_proj.lora_B)\n    comp = LatentCompressor(16, 2, n_heads=2)\n    save_trainable(tmp_path / "t.pt", m, {"compressor": comp})\n    m2 = nn.Sequential()\n    m2.add_module("q_proj", LoRALinear(nn.Linear(8, 8), r=2))\n    comp2 = LatentCompressor(16, 2, n_heads=2)\n    load_trainable(tmp_path / "t.pt", m2, {"compressor": comp2})\n    torch.testing.assert_close(m2.q_proj.lora_B, m.q_proj.lora_B)\n    torch.testing.assert_close(comp2.queries, comp.queries)\n\n\n# ------------------------------------------------------------------------------------------------------- train steps\n@pytest.mark.parametrize("mode,extra", [("none", {}), ("cot", {}), ("pause", {}),\n                                        ("latent", {"curriculum": True, "steps_per_stage": 1}),\n                                        ("latent", {"train_base": True, "lora_rank": 0, "grad_checkpointing": True,\n                                                    "bptt": 1})])\ndef test_one_train_step_per_mode(tmp_path, mode, extra):\n    cfg = dict(model="tiny", mode=mode, R=2, K=2, hops_train="2-3", hops_test="2,4", distractors=1, steps=3,\n               batch_size=2, n_test=2, eval_R_sweep="0,1", tiny_hidden=64, tiny_layers=2, tiny_intermediate=128,\n               tiny_heads=2, tiny_kv_heads=1, lora_rank=4, warmup_steps=1, log_every=1, out=str(tmp_path),\n               eval_every=2, eval_quick_n=2, save_weights=True)\n    cfg.update(extra)\n    res = train_latent.run(cfg)\n    res.pop("_reasoner")\n    assert len(res["loss_curve"]) == 3 and all(torch.isfinite(torch.tensor(x["loss"])) for x in res["loss_curve"])\n    exp_R = {"0"} if mode in ("none", "cot") else {"0", "1", "2"}\n    assert set(res["final_acc"]) == exp_R and set(res["final_acc"]["0"]) == {"2", "4"}\n    assert (tmp_path / "results.json").exists() and (tmp_path / "trainable.pt").exists()\n    if extra.get("curriculum"):\n        assert [x["stage"] for x in res["loss_curve"]] == [0, 1, 2]\n',
}

if USE_EMBEDDED_SOURCES or not _have_clone:
    for _path, _text in EMBEDDED_SOURCES.items():
        os.makedirs(os.path.dirname(_path), exist_ok=True)
        with open(_path, "w", encoding="utf-8") as _f:
            _f.write(_text)
    print(f"wrote {len(EMBEDDED_SOURCES)} embedded source files into {os.getcwd()}")
else:
    print("using the cloned sources in", os.getcwd(), "(set USE_EMBEDDED_SOURCES = True to overwrite them)")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

## Unit tests

In [ ]:
# CPU unit tests (~1-2 min). On Colab the tiny model uses the 2B4T tokenizer from the hub (same Llama-3 vocab).
!python -m pytest bitnet_vl/tests/test_latent.py -q -x

## Tiny sanity run

In [ ]:
import gc, json, os, time
import torch
from bitnet_vl.latent.train_latent import run

LOCAL_CHECK = bool(os.environ.get("LATENT_NB_LOCAL_CHECK"))  # set only by build_colab.py --check (CPU smoke run)
HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    # bf16 everywhere torch supports it (native on L4/A100, emulated on T4). 2B-4T overflows fp16: on a T4 the
    # fp16 smoke run gave NaN loss from step 1, bf16 trained normally (6.1 GB peak, ~1.2-1.5 ex/s).
    DTYPE = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
else:
    DTYPE = "fp32"
print("device:", torch.cuda.get_device_name() if HAS_CUDA else "cpu", "| dtype:", DTYPE)
RUN_DIR = "runs"
os.makedirs(RUN_DIR, exist_ok=True)


def run_and_free(cfg):
    res = run(cfg)
    res.pop("_reasoner", None)
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    return res

In [ ]:
# Tiny sanity run: random-init tiny BitNet (hidden 256, 4 layers), ALL weights trained (a random LM cannot do anything
# with LoRA only). Only checks that the pipeline trains end to end; accuracy of a tiny model is not a result.
tiny = run_and_free(dict(
    model="tiny", train_base=True, lora_rank=0, mode="latent", R=2, K=2, hops_train="2-3", hops_test="2-3",
    distractors=1, steps=20 if LOCAL_CHECK else 400, batch_size=8 if LOCAL_CHECK else 32, lr_base=1e-3,
    n_test=8 if LOCAL_CHECK else 100, eval_R_sweep="0,2", log_every=10 if LOCAL_CHECK else 50,
    dtype="fp32" if not HAS_CUDA else DTYPE, out=f"{RUN_DIR}/tiny_latent",
    **(dict(tiny_hidden=64, tiny_layers=2, tiny_intermediate=128) if LOCAL_CHECK else {})))
print("loss first/last:", tiny["loss_curve"][0]["loss"], tiny["loss_curve"][-1]["loss"])
print("final acc:", tiny["final_acc"])

## Main experiment: `microsoft/bitnet-b1.58-2B-4T-bf16`
Pick `PRESET`. Each mode reloads the frozen base (from the HF cache after the first download).

In [ ]:
MODEL = "microsoft/bitnet-b1.58-2B-4T-bf16"
PRESET = "QUICK"   # "QUICK" (~1 h on L4, fits a 16 GB T4 in bf16) or "FULL" (~5-7 h on L4); estimates
MODES = ["none", "cot", "pause", "latent"]

PRESETS = {
    # R thought steps x K soft tokens; curriculum: R+1 stages of `steps_per_stage` (stage 0 = CoT, last = all latent)
    "QUICK": dict(R=4, K=4, hops_train="2-4", hops_test="2-6", distractors=1, steps=500, batch_size=4, grad_accum=4,
                  steps_per_stage=50, n_test=50, eval_every=0, eval_R_sweep="0,1,2,4,6,8"),
    "FULL": dict(R=6, K=4, hops_train="2-6", hops_test="2-8", distractors=2, steps=2000, batch_size=8, grad_accum=4,
                 steps_per_stage=150, n_test=200, eval_every=500, eval_R_sweep="0,1,2,4,6,8,10"),
}
COMMON_CFG = dict(model=MODEL, dtype=DTYPE, lora_rank=16, lora_alpha=32, lr_lora=2e-4, lr_compressor=1e-3,
                  warmup_steps=30, grad_checkpointing=True, log_every=25, eval_batch_size=32, seed=0)

if LOCAL_CHECK:  # CPU smoke test of this very cell with the tiny model (used by build_colab.py --check)
    MODEL = "tiny"
    COMMON_CFG.update(model="tiny", train_base=True, lora_rank=0, lr_base=1e-3, dtype="fp32",
                      tiny_hidden=64, tiny_layers=2, tiny_intermediate=128, grad_checkpointing=True)
    PRESETS["QUICK"].update(R=2, K=2, hops_train="2-3", hops_test="2-4", steps=6, batch_size=2, grad_accum=1,
                            steps_per_stage=2, n_test=4, eval_R_sweep="0,1,2,4")


def mode_cfg(mode):
    cfg = dict(COMMON_CFG, **PRESETS[PRESET], mode=mode, out=f"{RUN_DIR}/{PRESET.lower()}_{mode}")
    cfg["curriculum"] = mode in ("latent", "pause")
    return cfg

for m in MODES:
    print(m, mode_cfg(m))

In [ ]:
results = {}
for mode in MODES:
    t0 = time.time()
    print(f"===== {mode} =====")
    results[mode] = run_and_free(mode_cfg(mode))
    tp = results[mode]["throughput"]
    print(f"{mode}: {time.time() - t0:.0f}s, {tp['examples_per_s']:.1f} ex/s, "
          f"max mem {results[mode]['max_memory_gb']} GB, acc {results[mode]['final_acc']}")
json.dump({k: v for k, v in results.items()}, open(f"{RUN_DIR}/{PRESET.lower()}_all.json", "w"), indent=1)

### Eval-R sweep (test-time scaling)
`final_acc[R][hops]` is measured with the all-latent setting and R thought steps at test time (R may differ from the
training R; R = 0 means the answer is decoded right after the prompt). `none` / `cot` have a single entry.

In [ ]:
import pandas as pd

rows = []
for mode, res in results.items():
    for R, by_h in res["final_acc"].items():
        for h, acc in by_h.items():
            rows.append(dict(mode=mode, eval_R=int(R), hops=int(h), acc=acc))
df = pd.DataFrame(rows)
train_R = PRESETS[PRESET]["R"]
print("accuracy at the training R (none/cot: R=0), rows = mode, cols = hops")
main = df[(df.eval_R == train_R) | df["mode"].isin(["none", "cot"])]
display(main.pivot_table(index="mode", columns="hops", values="acc"))
print("eval-R sweep, mean over test hops")
display(df[df["mode"].isin(["latent", "pause"])].pivot_table(index="mode", columns="eval_R", values="acc"))

## Plots

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for mode, res in results.items():
    R = str(train_R) if str(train_R) in res["final_acc"] else "0"
    hs = sorted(res["final_acc"][R], key=int)
    ax[0].plot([int(h) for h in hs], [res["final_acc"][R][h] for h in hs], marker="o", label=mode)
lo, hi = PRESETS[PRESET]["hops_train"].split("-")
ax[0].axvspan(int(lo), int(hi), color="grey", alpha=0.12, label="train hops")
ax[0].set(xlabel="hops", ylabel="accuracy", title=f"accuracy vs hops (eval R = train R = {train_R})", ylim=(0, 1.02))
ax[0].legend()
for mode in ("latent", "pause"):
    if mode in results:
        fa = results[mode]["final_acc"]
        Rs = sorted(fa, key=int)
        ax[1].plot([int(r) for r in Rs], [sum(fa[r].values()) / len(fa[r]) for r in Rs], marker="o", label=mode)
for mode in ("none", "cot"):
    if mode in results:
        a = results[mode]["final_acc"]["0"]
        ax[1].axhline(sum(a.values()) / len(a), ls="--", lw=1, label=mode, color="C0" if mode == "none" else "C1")
ax[1].axvline(train_R, color="grey", ls=":", lw=1)
ax[1].set(xlabel="eval R (thought steps at test time)", ylabel="mean accuracy over test hops",
          title="test-time scaling", ylim=(0, 1.02))
ax[1].legend()
for mode, res in results.items():
    lc = res["loss_curve"]
    ax[2].plot([x["step"] for x in lc], [x["loss"] for x in lc], label=mode)
ax[2].set(xlabel="optimizer step", ylabel="train loss", yscale="log", title="training loss")
ax[2].legend()
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/{PRESET.lower()}_plots.png", dpi=120)
plt.show()

## Save to Google Drive (optional)

In [ ]:
# Optional: copy runs/ (results.json per mode + plots) to Google Drive.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/bitnet_latent_reasoning/{time.strftime('%Y%m%d-%H%M%S')}"
    shutil.copytree(RUN_DIR, dst)
    print("saved to", dst)